# Multicamera Pose Exercise Observability Fusion

Чистая GitHub-версия финального решения **без дополнительных моделей инвентаря**.

Что делает ноутбук:

1. берет несколько синхронных камер одного выполнения упражнения;
2. режет нужные интервалы видео;
3. запускает YOLO Pose и получает скелет: keypoints + confidence;
4. при необходимости автоматически делает ROI по активному спортсмену, но потом возвращает точки в координаты исходного кадра;
5. считает весь набор pose-фич упражнения: ноги, колени, корпус, руки, голова, скорость и направление движения;
6. объединяет одинаковые фичи между камерами покадрово через confidence/observability веса;
7. сравнивает fused-выполнение с reference-прокатами `prokat_*`;
8. отдельно считает coverage/visibility factor, чтобы плохая видимость нужных точек снижала финальный score.

Важно: графики показывают только несколько понятных фич для визуального контроля, но score считается по всем pose-фичам из `az.PHASE_CHANNELS` и `az.QUALITY_CHANNELS`.


## Что считает метрика

`raw_similarity_score` — это **reference-based similarity score**: насколько выполнение похоже на размеченные reference-прокаты.

Reference сейчас такие:

- `prokat_1` — хороший прокат;
- `prokat_2` — торопится;
- `prokat_3` — слишком размахивает руками;
- `prokat_4` — плохой прокат.

Упрощенно:

```text
video -> pose -> feature curves -> distance to prokat_* -> raw_similarity_score 0..100
```

`score` — финальный score после учета видимости:

```text
score = raw_similarity_score * coverage_factor
```

Где `coverage_factor` в диапазоне `0..1` показывает, насколько нужные pose-фичи реально были видны. Если модель плохо видит точки для важных фич, финальная оценка снижается, даже если оставшиеся видимые фичи похожи на эталон.


## Как работает multicamera fusion

Для каждой фичи и каждого момента времени объединяем значения с камер не поровну, а по весу:

```text
fused_feature[t] =
    sum(camera_feature[t] * camera_weight[t]) /
    sum(camera_weight[t])
```

Вес камеры считается отдельно для каждой фичи и каждого кадра:

```text
camera_weight[t] = keypoint_confidence[t] * observability[t]
```

- `keypoint_confidence` — насколько YOLO Pose уверена в тех точках тела, из которых считается конкретная фича.
- `observability` — насколько эта фича читается из текущей 2D-геометрии скелета. Например, для ног важна читаемость бедро-колено-голеностоп, для корпуса — плечи и таз, для рук — плечо-локоть-запястье.
- `confidence threshold` отсекает кадры, где нужные точки почти не видны.

Нет ручных правил вида `cam1 = front`, `cam2 = side`. Камера получает больший вклад только если в этот момент она лучше видит точки, нужные именно для этой фичи.


## Как читать графики

На графиках слева:

- цветные линии — значения одной фичи с отдельных камер;
- черная линия — итоговая fused-фича после объединения камер;
- `x` — синхронное время в секундах;
- `y` — значение конкретной фичи.

На графиках справа:

- относительный вклад каждой камеры в fused-сигнал;
- `1.0` — почти весь сигнал взят из этой камеры;
- `~0.33` — три камеры влияют примерно поровну;
- `0` — камера почти не участвует, потому что нужные точки не видны или не проходят confidence threshold.

Основные визуальные фичи:

- `ankles_distance`: чем выше, тем лодыжки дальше друг от друга;
- `torso_lean_abs_deg`: чем выше, тем сильнее наклон корпуса.

Это диагностические графики. Итоговый score ниже считается по полному набору pose-фич, а не только по этим двум линиям.


In [ ]:
!pip install ultralytics dtaidistance


In [ ]:
import pathlib
import re
import shutil
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Literal, Optional

import cv2
import dtaidistance.dtw as dtw_mod
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, medfilt
from ultralytics import YOLO

warnings.filterwarnings("ignore")

ROOT = pathlib.Path(r"c:\Projects\hockey")
VIDEO_DIR = ROOT / "videos"
OUTPUT_DIR = ROOT / "outputs" / "pose_compare_v4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_VIDEO_NAMES = ["prokat_1", "prokat_2", "prokat_3", "prokat_4"]
VIDEOS = {name: VIDEO_DIR / f"{name}.mp4" for name in REFERENCE_VIDEO_NAMES}
REFERENCE_LEVELS = {
    "prokat_1": 1.00,
    "prokat_2": 0.67,
    "prokat_3": 0.33,
    "prokat_4": 0.00,
}

MANUAL_QUALITY_WEIGHTS_V4 = {
    "head_up": 3.5,
    "torso_upright": 3.5,
    "upper_body_stability": 2.8,
    "peak_posture_consistency": 2.5,
    "hands_compact": 1.8,
    "arm_discipline": 1.8,
    "knee_symmetry": 1.5,
    "tempo_stability": 1.3,
    "cycle_amplitude_cv": 1.0,
    "leg_control": 1.0,
    "direction_control": 2.8,
    "body_motion_alignment": 3.0,
    "head_motion_alignment": 2.8,
    "head_body_heading_consistency": 2.2,
}

MANUAL_PHASE_WEIGHTS_V4 = {
    "ankles_distance": 2.4,
    "ankles_distance_vel": 1.4,
    "torso_lean_abs_deg": 2.8,
    "torso_lean_abs_deg_vel": 1.3,
    "head_pitch_rel": 2.8,
    "head_pitch_rel_vel": 1.3,
    "spine_angle": 1.8,
    "spine_angle_vel": 0.8,
    "knee_angle_mean": 1.3,
    "knee_angle_asym": 1.5,
    "hip_lateral_sway": 1.2,
    "hands_to_torso_mean": 1.3,
    "motion_speed_norm": 2.0,
    "motion_speed_norm_vel": 1.0,
    "motion_dir_stability_local": 1.8,
    "motion_dir_stability_local_vel": 0.8,
    "body_motion_align": 2.8,
    "body_motion_align_vel": 1.2,
    "head_motion_align": 2.6,
    "head_motion_align_vel": 1.1,
}

TRACKER_CONFIG = {
    "min_bbox_area_ratio": 0.012,
    "min_pose_score": 0.18,
    "area_weight": 0.35,
    "center_weight": 0.12,
    "pose_weight": 0.18,
    "track_weight": 0.25,
    "motion_weight": 0.10,
    "center_radius_ratio": 0.55,
    "max_center_jump_ratio": 0.22,
    "motion_norm_ratio": 0.12,
    "max_track_misses": 6,
}

ANALYZER_CONFIG = {
    "conf_thr": 0.25,
    "infer_imgsz": 1920,
    "infer_device": "0",
    "smooth_window": 9,
    "outlier_thr": 3.0,
    "alpha": 0.42,
    "mode": "manual",
    "phase_weights": MANUAL_PHASE_WEIGHTS_V4,
    "quality_weights": MANUAL_QUALITY_WEIGHTS_V4,
    "reference_scores": REFERENCE_LEVELS,
    "tracker": TRACKER_CONFIG,
    "weights_path": "yolo26m-pose.pt",
    "reference_temperature_scale": 0.35,
    "force_rebuild_reference_cache": False,
}

print("Output dir:", OUTPUT_DIR)
print("Reference videos:", {k: p.exists() for k, p in VIDEOS.items()})
print("Reference scores:", REFERENCE_LEVELS)
print("Tracker config:", TRACKER_CONFIG)


In [ ]:
# Kaggle/local path setup. Run after the config cell.
KAGGLE_DATA_ROOT = pathlib.Path("/kaggle/input/datasets/alpin0s/raspoznavashka-data/raspoznavashka-data")
LOCAL_DATA_ROOT = pathlib.Path.cwd() / "raspoznavashka-data"

WORK_ROOT = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
DATA_ROOT = KAGGLE_DATA_ROOT if KAGGLE_DATA_ROOT.exists() else (LOCAL_DATA_ROOT if LOCAL_DATA_ROOT.exists() else pathlib.Path.cwd())
ROOT = WORK_ROOT
VIDEO_DIR = DATA_ROOT
OUTPUT_DIR = WORK_ROOT / "outputs" / "pose_compare_v4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_VIDEO_NAMES = ["prokat_1", "prokat_2", "prokat_3", "prokat_4"]
VIDEOS = {name: VIDEO_DIR / f"{name}.mp4" for name in REFERENCE_VIDEO_NAMES}

MODEL_CANDIDATES = [
    DATA_ROOT / "yolo26m-pose.pt",
    WORK_ROOT / "yolo26m-pose.pt",
    pathlib.Path.cwd() / "yolo26m-pose.pt",
]


model_path = next((p for p in MODEL_CANDIDATES if p.exists()), pathlib.Path("yolo26m-pose.pt"))
ANALYZER_CONFIG["weights_path"] = str(model_path)
ANALYZER_CONFIG["infer_imgsz"] = 1920
ANALYZER_CONFIG["infer_device"] = "0"

print("WORK_ROOT:", WORK_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("VIDEO_DIR:", VIDEO_DIR)
print("Output dir:", OUTPUT_DIR)
print("Reference videos:", {k: p.exists() for k, p in VIDEOS.items()})
print("weights_path:", ANALYZER_CONFIG["weights_path"])
print("infer_imgsz:", ANALYZER_CONFIG["infer_imgsz"])
print("infer_device:", ANALYZER_CONFIG["infer_device"])

try:
    import torch
    print("CUDA available:", bool(torch.cuda.is_available()))
    print("CUDA device_count:", int(torch.cuda.device_count()) if torch.cuda.is_available() else 0)
except Exception as exc:
    print("CUDA check failed:", repr(exc))


In [ ]:
KP = dict(
    nose=0, l_eye=1, r_eye=2, l_ear=3, r_ear=4,
    l_shoulder=5, r_shoulder=6, l_elbow=7, r_elbow=8,
    l_wrist=9, r_wrist=10, l_hip=11, r_hip=12,
    l_knee=13, r_knee=14, l_ankle=15, r_ankle=16,
)
LR_PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]
SKELETON_EDGES = [
    (KP["nose"], KP["l_shoulder"]), (KP["nose"], KP["r_shoulder"]),
    (KP["l_shoulder"], KP["r_shoulder"]),
    (KP["l_shoulder"], KP["l_elbow"]), (KP["l_elbow"], KP["l_wrist"]),
    (KP["r_shoulder"], KP["r_elbow"]), (KP["r_elbow"], KP["r_wrist"]),
    (KP["l_shoulder"], KP["l_hip"]), (KP["r_shoulder"], KP["r_hip"]),
    (KP["l_hip"], KP["r_hip"]),
    (KP["l_hip"], KP["l_knee"]), (KP["l_knee"], KP["l_ankle"]),
    (KP["r_hip"], KP["r_knee"]), (KP["r_knee"], KP["r_ankle"]),
]
LOWER_BODY_IDS = [KP["l_hip"], KP["r_hip"], KP["l_knee"], KP["r_knee"], KP["l_ankle"], KP["r_ankle"]]
TORSO_IDS = [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]]


def _to_float(x):
    return float(np.asarray(x).ravel()[0])


@dataclass
class Cfg:
    conf_thr: float = 0.25
    infer_imgsz: int = 1920
    infer_device: str = "0"
    smooth_window: int = 9
    outlier_thr: float = 3.0
    alpha: float = 0.40
    mode: Literal["auto", "manual"] = "manual"
    phase_weights: Dict[str, float] = field(default_factory=dict)
    quality_weights: Dict[str, float] = field(default_factory=dict)
    reference_scores: Dict[str, float] = field(default_factory=dict)
    tracker: Dict[str, float] = field(default_factory=dict)
    weights_path: str = "yolo26m-pose.pt"
    reference_temperature_scale: float = 0.35
    force_rebuild_reference_cache: bool = False


class Analyzer:
    """Анализ техники хоккейного упражнения с трекингом исполнителя и reference-based scoring."""

    BASE_PHASE = [
        "ankles_distance",
        "knee_angle_mean",
        "knee_angle_asym",
        "torso_lean_abs_deg",
        "hands_to_torso_mean",
        "head_pitch_rel",
        "hip_lateral_sway",
        "elbow_angle_mean",
        "shoulder_elevation",
        "spine_angle",
        "motion_speed_norm",
        "motion_dir_stability_local",
        "body_motion_align",
        "head_motion_align",
    ]

    QUALITY_CHANNELS = [
        "head_up",
        "torso_upright",
        "hands_compact",
        "tempo_stability",
        "leg_control",
        "knee_symmetry",
        "upper_body_stability",
        "arm_discipline",
        "cycle_amplitude_cv",
        "peak_posture_consistency",
        "direction_control",
        "body_motion_alignment",
        "head_motion_alignment",
        "head_body_heading_consistency",
    ]

    @property
    def PHASE_CHANNELS(self):
        return self.BASE_PHASE + [f"{c}_vel" for c in self.BASE_PHASE]

    def __init__(self, cfg: Optional[Cfg] = None):
        self.cfg = cfg or Cfg()
        self.model: Optional[YOLO] = None
        self._pose_cache: Dict[str, dict] = {}
        self._phase_w: Optional[np.ndarray] = None
        self._quality_w: Optional[np.ndarray] = None

    @staticmethod
    def _hex(rgb: np.ndarray) -> str:
        rgb = np.clip(np.round(rgb).astype(int), 0, 255)
        return f"#{rgb[0]:02x}{rgb[1]:02x}{rgb[2]:02x}"

    def reference_color(self, name: str) -> str:
        if name not in self.cfg.reference_scores:
            return "#3498db"
        low = np.array([231, 76, 60], dtype=float)
        high = np.array([46, 204, 113], dtype=float)
        value = float(self.cfg.reference_scores[name])
        return self._hex(low * (1.0 - value) + high * value)

    def _reference_names(self, features: dict) -> List[str]:
        return [name for name in self.cfg.reference_scores if name in features]

    def load_model(self, weights: Optional[str] = None):
        self.model = YOLO(weights or self.cfg.weights_path)

    def _resolve_infer_device(self):
        return str(getattr(self.cfg, "infer_device", "0"))

    def _cache_path(self, name: str) -> pathlib.Path:
        return OUTPUT_DIR / f"{name}_pose.npz"

    def _video_path_for(self, name: str) -> pathlib.Path:
        if name in self._pose_cache and "video_path" in self._pose_cache[name]:
            return pathlib.Path(str(self._pose_cache[name]["video_path"]))
        return VIDEOS.get(name, VIDEO_DIR / f"{name}.mp4")

    @staticmethod
    def _bbox_from_keypoints(kp: np.ndarray) -> np.ndarray:
        visible = kp[:, 2] > 0
        if not np.any(visible):
            return np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)
        xy = kp[visible, :2]
        x1, y1 = xy.min(axis=0)
        x2, y2 = xy.max(axis=0)
        return np.array([x1, y1, x2, y2], dtype=np.float32)

    @staticmethod
    def _bbox_area(box: np.ndarray) -> float:
        return float(max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1]))

    @staticmethod
    def _bbox_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
        xa1, ya1, xa2, ya2 = box_a
        xb1, yb1, xb2, yb2 = box_b
        inter_w = max(0.0, min(xa2, xb2) - max(xa1, xb1))
        inter_h = max(0.0, min(ya2, yb2) - max(ya1, yb1))
        inter = inter_w * inter_h
        union = Analyzer._bbox_area(box_a) + Analyzer._bbox_area(box_b) - inter + 1e-8
        return float(inter / union)

    @staticmethod
    def _torso_center(xy: np.ndarray, conf: np.ndarray) -> np.ndarray:
        w = np.clip(conf[TORSO_IDS], 0.0, 1.0)
        wsum = w.sum() + 1e-6
        if wsum <= 1e-6:
            return xy.mean(axis=0)
        return (xy[TORSO_IDS] * w[:, None]).sum(axis=0) / wsum

    def _candidate_score(self, frame_shape, xy: np.ndarray, conf: np.ndarray, box: np.ndarray, prev_state: Optional[dict]) -> dict:
        cfg = self.cfg.tracker
        frame_h, frame_w = frame_shape[:2]
        frame_area = float(frame_h * frame_w)
        frame_diag = float(np.hypot(frame_w, frame_h)) + 1e-6
        frame_center = np.array([frame_w / 2.0, frame_h / 2.0], dtype=np.float32)

        area_ratio = self._bbox_area(box) / (frame_area + 1e-8)
        mean_conf = float(np.mean(conf))
        lower_conf = float(np.mean(conf[LOWER_BODY_IDS]))
        torso_center = self._torso_center(xy, conf)

        center_dist_ratio = float(np.linalg.norm(torso_center - frame_center) / frame_diag)
        center_score = max(0.0, 1.0 - center_dist_ratio / max(cfg["center_radius_ratio"], 1e-6))
        pose_score = 0.65 * mean_conf + 0.35 * lower_conf
        area_score = min(1.0, np.sqrt(max(area_ratio, 0.0)) / 0.35)

        static_score = (
            cfg["area_weight"] * area_score
            + cfg["center_weight"] * center_score
            + cfg["pose_weight"] * pose_score
        )

        track_score = 0.0
        motion_score = 0.0
        if prev_state is not None:
            prev_center = prev_state["center"]
            prev_box = prev_state["box"]
            jump_ratio = float(np.linalg.norm(torso_center - prev_center) / frame_diag)
            jump_score = np.exp(-((jump_ratio / max(cfg["max_center_jump_ratio"], 1e-6)) ** 2))
            iou = self._bbox_iou(box, prev_box)
            track_score = 0.55 * jump_score + 0.45 * iou
            motion_score = min(1.0, jump_ratio / max(cfg["motion_norm_ratio"], 1e-6))

        penalty = 0.0
        if area_ratio < cfg["min_bbox_area_ratio"]:
            penalty += 0.35
        if pose_score < cfg["min_pose_score"]:
            penalty += 0.35

        total_score = static_score + cfg["track_weight"] * track_score + cfg["motion_weight"] * motion_score - penalty
        return {
            "score": float(total_score),
            "center": torso_center.astype(np.float32),
            "area_ratio": float(area_ratio),
            "pose_score": float(pose_score),
            "track_score": float(track_score),
            "motion_score": float(motion_score),
        }

    def _select_primary_athlete(self, frame_shape, xy_all: np.ndarray, conf_all: np.ndarray, boxes: np.ndarray, prev_state: Optional[dict]):
        metas = [self._candidate_score(frame_shape, xy_all[i], conf_all[i], boxes[i], prev_state) for i in range(len(xy_all))]
        best_idx = int(np.argmax([m["score"] for m in metas]))
        return best_idx, metas[best_idx]

    def _load_cached(self, name: str, npz: pathlib.Path) -> Optional[dict]:
        raw = dict(np.load(npz, allow_pickle=True))
        if "kps" not in raw and "kps_xy" in raw:
            xy = raw["kps_xy"]
            cf = raw["kps_conf"]
            raw = {"kps": np.concatenate([xy, cf[:, :, None]], axis=2), "fps": raw["fps"]}
        has_tracking = all(key in raw for key in ("track_boxes", "track_scores", "candidate_counts", "video_path"))
        if not has_tracking:
            return None
        cached_imgsz = int(_to_float(raw["infer_imgsz"])) if "infer_imgsz" in raw else None
        if cached_imgsz != int(self.cfg.infer_imgsz):
            return None
        return {
            **raw,
            "fps": np.float32(_to_float(raw["fps"])),
            "video_path": str(np.asarray(raw["video_path"]).reshape(-1)[0]),
        }

    def run_inference(self, name: str, video_path, force: bool = False) -> dict:
        video_path = pathlib.Path(video_path)
        npz = self._cache_path(name)
        if npz.exists() and not force:
            cached = self._load_cached(name, npz)
            if cached is not None:
                self._pose_cache[name] = cached
                print(f"  {name}: {cached['kps'].shape[0]} frames (cached with tracking)")
                return cached
            print(f"  {name}: legacy cache without athlete tracking -> re-running inference")

        if self.model is None:
            self.load_model()

        infer_device = self._resolve_infer_device()
        print(f"  {name}: inference device={infer_device}, imgsz={self.cfg.infer_imgsz}")

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video: {video_path}")

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        kps_list = []
        track_boxes = []
        track_scores = []
        candidate_counts = []
        prev_state = None
        miss_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            res = self.model(frame, verbose=False, conf=self.cfg.conf_thr, imgsz=self.cfg.infer_imgsz, device=infer_device)
            result = res[0]

            if result.keypoints is not None and len(result.keypoints.xy) > 0:
                xy_all = result.keypoints.xy.cpu().numpy()
                if result.keypoints.conf is not None:
                    conf_all = result.keypoints.conf.cpu().numpy()
                else:
                    conf_all = np.ones((len(xy_all), len(KP)), dtype=np.float32)

                if getattr(result, "boxes", None) is not None and len(result.boxes) > 0:
                    boxes = result.boxes.xyxy.cpu().numpy().astype(np.float32)
                else:
                    boxes = np.array([
                        self._bbox_from_keypoints(np.concatenate([xy_all[i], conf_all[i][:, None]], axis=1))
                        for i in range(len(xy_all))
                    ], dtype=np.float32)

                chosen_idx, meta = self._select_primary_athlete(frame.shape, xy_all, conf_all, boxes, prev_state)
                chosen_kp = np.concatenate([xy_all[chosen_idx], conf_all[chosen_idx][:, None]], axis=1).astype(np.float32)
                chosen_box = boxes[chosen_idx].astype(np.float32)

                kps_list.append(chosen_kp)
                track_boxes.append(chosen_box)
                track_scores.append(meta["score"])
                candidate_counts.append(len(xy_all))

                prev_state = {"box": chosen_box, "center": meta["center"]}
                miss_count = 0
            else:
                kps_list.append(np.zeros((17, 3), dtype=np.float32))
                track_boxes.append(np.array([np.nan, np.nan, np.nan, np.nan], dtype=np.float32))
                track_scores.append(np.nan)
                candidate_counts.append(0)
                miss_count += 1
                if miss_count >= int(self.cfg.tracker["max_track_misses"]):
                    prev_state = None

        cap.release()

        data = {
            "kps": np.asarray(kps_list, dtype=np.float32),
            "track_boxes": np.asarray(track_boxes, dtype=np.float32),
            "track_scores": np.asarray(track_scores, dtype=np.float32),
            "candidate_counts": np.asarray(candidate_counts, dtype=np.int16),
            "fps": np.float32(fps),
            "infer_imgsz": np.int32(self.cfg.infer_imgsz),
            "video_path": np.array(str(video_path)),
        }
        np.savez_compressed(npz, **data)
        self._pose_cache[name] = {**data, "video_path": str(video_path)}
        print(f"  {name}: {data['kps'].shape[0]} frames, fps={fps:.1f} (tracked inference)")
        return self._pose_cache[name]

    def normalize(self, kps: np.ndarray) -> np.ndarray:
        kps = kps.copy()
        ls = kps[:, KP["l_shoulder"], :2]
        rs = kps[:, KP["r_shoulder"], :2]
        lh = kps[:, KP["l_hip"], :2]
        rh = kps[:, KP["r_hip"], :2]
        centre = (ls + rs + lh + rh) / 4
        kps[:, :, :2] -= centre[:, None, :]

        torso = np.linalg.norm(((ls + rs) / 2) - ((lh + rh) / 2), axis=1)
        sw = np.linalg.norm(ls - rs, axis=1)
        hw = np.linalg.norm(lh - rh, axis=1)
        scale = np.median(np.stack([torso, sw, hw], axis=1), axis=1)
        scale = np.where(scale < 1e-3, 1.0, scale)
        kps[:, :, :2] /= scale[:, None, None]
        return kps

    @staticmethod
    def _angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
        ba = a - b
        bc = c - b
        denom = np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-8
        cos = np.einsum("...i,...i", ba, bc) / denom
        return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

    @staticmethod
    def _smooth(x: np.ndarray, w: int = 9) -> np.ndarray:
        k = max(3, w if w % 2 == 1 else w + 1)
        if len(x) < k:
            return x.copy()
        return medfilt(x, kernel_size=k)

    def _clean(self, x: np.ndarray) -> np.ndarray:
        thr = self.cfg.outlier_thr
        med = np.median(x)
        iqr = np.percentile(x, 75) - np.percentile(x, 25)
        mask = np.abs(x - med) > thr * (iqr + 1e-8)
        out = x.copy()
        out[mask] = med
        return out

    @staticmethod
    def _z(x: np.ndarray) -> np.ndarray:
        return (x - x.mean()) / (x.std() + 1e-8)

    @staticmethod
    def _unit(v: np.ndarray) -> np.ndarray:
        n = np.linalg.norm(v, axis=1, keepdims=True) + 1e-8
        return v / n

    def _motion_meta(self, kps_raw: np.ndarray) -> dict:
        xy = kps_raw[:, :, :2]
        cf = kps_raw[:, :, 2]

        torso_ids = np.array(TORSO_IDS)
        w = np.clip(cf[:, torso_ids], 0, 1)
        wsum = w.sum(axis=1, keepdims=True) + 1e-6
        com = (xy[:, torso_ids, :] * w[:, :, None]).sum(axis=1) / wsum

        com_x = self._smooth(com[:, 0], self.cfg.smooth_window)
        com_y = self._smooth(com[:, 1], self.cfg.smooth_window)
        com_s = np.stack([com_x, com_y], axis=1)
        vel = np.gradient(com_s, axis=0)

        mid_sh = (xy[:, KP["l_shoulder"], :] + xy[:, KP["r_shoulder"], :]) / 2
        mid_hp = (xy[:, KP["l_hip"], :] + xy[:, KP["r_hip"], :]) / 2
        torso_px = np.linalg.norm(mid_sh - mid_hp, axis=1) + 1e-6
        speed_px = np.linalg.norm(vel, axis=1)
        speed_norm = speed_px / torso_px

        dir_u = np.zeros_like(vel)
        prev = np.array([1.0, 0.0], dtype=np.float32)
        for i in range(len(vel)):
            if speed_norm[i] > 0.02:
                prev = vel[i] / (np.linalg.norm(vel[i]) + 1e-8)
            dir_u[i] = prev

        sh_l = xy[:, KP["l_shoulder"], :]
        sh_r = xy[:, KP["r_shoulder"], :]
        shoulder_v = sh_r - sh_l
        n1 = np.stack([-shoulder_v[:, 1], shoulder_v[:, 0]], axis=1)
        n2 = -n1
        nose = xy[:, KP["nose"], :]
        nose_hint = nose - (sh_l + sh_r) / 2
        pick_n1 = np.sum(n1 * nose_hint, axis=1) >= 0
        body_heading = np.where(pick_n1[:, None], n1, n2)
        body_u = self._unit(body_heading)

        head_v = nose - (sh_l + sh_r) / 2
        head_u = self._unit(head_v)

        body_align = np.sum(body_u * dir_u, axis=1)
        head_align = np.sum(head_u * dir_u, axis=1)
        head_body_cons = np.sum(body_u * head_u, axis=1)

        motion_ang = np.unwrap(np.arctan2(dir_u[:, 1], dir_u[:, 0]))
        local_std = np.zeros_like(motion_ang)
        half = max(2, self.cfg.smooth_window // 2)
        for i in range(len(motion_ang)):
            left = max(0, i - half)
            right = min(len(motion_ang), i + half + 1)
            local_std[i] = np.std(motion_ang[left:right])
        motion_dir_stability_local = -local_std

        return {
            "com_px": com_s,
            "motion_vec_px": vel,
            "motion_dir_u": dir_u,
            "speed_norm": speed_norm,
            "body_align": body_align,
            "head_align": head_align,
            "head_body_cons": head_body_cons,
            "motion_dir_stability_local": motion_dir_stability_local,
        }

    def extract(self, name: str) -> dict:
        data = self._pose_cache[name]
        kps_raw = data["kps"]
        fps = _to_float(data["fps"])
        kps = self.normalize(kps_raw)

        def pt(i):
            return kps[:, i, :2]

        mid_sh = (pt(KP["l_shoulder"]) + pt(KP["r_shoulder"])) / 2
        mid_hp = (pt(KP["l_hip"]) + pt(KP["r_hip"])) / 2
        mid_ank = (pt(KP["l_ankle"]) + pt(KP["r_ankle"])) / 2

        ankle_d = np.abs(pt(KP["l_ankle"])[:, 0] - pt(KP["r_ankle"])[:, 0])
        l_kang = self._angle(pt(KP["l_hip"]), pt(KP["l_knee"]), pt(KP["l_ankle"]))
        r_kang = self._angle(pt(KP["r_hip"]), pt(KP["r_knee"]), pt(KP["r_ankle"]))

        torso_v = mid_sh - mid_hp
        torso_lean = np.degrees(np.arctan2(np.abs(torso_v[:, 0]), np.abs(torso_v[:, 1]) + 1e-8))

        l_hd = np.linalg.norm(pt(KP["l_wrist"]) - pt(KP["l_hip"]), axis=1)
        r_hd = np.linalg.norm(pt(KP["r_wrist"]) - pt(KP["r_hip"]), axis=1)

        nose_v = pt(KP["nose"]) - mid_sh
        head_pitch = np.degrees(np.arctan2(-nose_v[:, 1], np.abs(nose_v[:, 0]) + 1e-8))

        hip_sway = mid_hp[:, 0] - mid_sh[:, 0]
        l_eang = self._angle(pt(KP["l_shoulder"]), pt(KP["l_elbow"]), pt(KP["l_wrist"]))
        r_eang = self._angle(pt(KP["r_shoulder"]), pt(KP["r_elbow"]), pt(KP["r_wrist"]))
        shoulder_elev = mid_sh[:, 1]
        spine_angle = self._angle(mid_sh, mid_hp, mid_ank)

        motion = self._motion_meta(kps_raw)

        raw_map = {
            "ankles_distance": ankle_d,
            "knee_angle_mean": (l_kang + r_kang) / 2,
            "knee_angle_asym": np.abs(l_kang - r_kang),
            "torso_lean_abs_deg": torso_lean,
            "hands_to_torso_mean": (l_hd + r_hd) / 2,
            "head_pitch_rel": head_pitch,
            "hip_lateral_sway": hip_sway,
            "elbow_angle_mean": (l_eang + r_eang) / 2,
            "shoulder_elevation": shoulder_elev,
            "spine_angle": spine_angle,
            "motion_speed_norm": motion["speed_norm"],
            "motion_dir_stability_local": motion["motion_dir_stability_local"],
            "body_motion_align": motion["body_align"],
            "head_motion_align": motion["head_align"],
        }

        phase = {}
        for ch, arr in raw_map.items():
            clean = self._clean(arr)
            phase[ch] = self._smooth(clean, self.cfg.smooth_window)
            phase[f"{ch}_vel"] = self._smooth(np.gradient(phase[ch]), self.cfg.smooth_window)

        dist = phase["ankles_distance"]
        min_dist = max(2, int(fps * 0.3))
        peaks, _ = find_peaks(dist, height=np.percentile(dist, 40), distance=min_dist)
        troughs, _ = find_peaks(-dist, height=np.percentile(-dist, 40), distance=min_dist)
        events = np.sort(np.unique(np.concatenate([peaks, troughs])))

        iei = np.diff(events) / fps if len(events) > 1 else np.array([1.0])
        peak_ankl = dist[events] if len(events) > 0 else dist
        peak_torso = phase["torso_lean_abs_deg"][events] if len(events) > 0 else phase["torso_lean_abs_deg"]
        ankl_mean = dist.mean() + 1e-8

        rel_mask = motion["speed_norm"] > np.percentile(motion["speed_norm"], 35)
        if rel_mask.sum() < 5:
            rel_mask = np.ones_like(rel_mask, dtype=bool)

        quality = {
            "head_up": -np.median(phase["head_pitch_rel"]),
            "torso_upright": -np.median(phase["torso_lean_abs_deg"]),
            "hands_compact": -np.median(phase["hands_to_torso_mean"]),
            "tempo_stability": -(iei.std() / (iei.mean() + 1e-8)),
            "leg_control": -(phase["ankles_distance"].std() / ankl_mean),
            "knee_symmetry": -np.median(phase["knee_angle_asym"]),
            "upper_body_stability": -phase["torso_lean_abs_deg"].std(),
            "arm_discipline": -phase["hands_to_torso_mean"].std(),
            "cycle_amplitude_cv": -(peak_ankl.std() / (peak_ankl.mean() + 1e-8)),
            "peak_posture_consistency": -peak_torso.std(),
            "direction_control": -np.std(np.unwrap(np.arctan2(motion["motion_dir_u"][rel_mask, 1], motion["motion_dir_u"][rel_mask, 0]))),
            "body_motion_alignment": np.median(motion["body_align"][rel_mask]),
            "head_motion_alignment": np.median(motion["head_align"][rel_mask]),
            "head_body_heading_consistency": np.median(motion["head_body_cons"][rel_mask]),
        }

        track_boxes = data.get("track_boxes", np.full((len(kps_raw), 4), np.nan, dtype=np.float32))
        valid_boxes = np.isfinite(track_boxes).all(axis=1)
        area_ratios = []
        if valid_boxes.any():
            width = np.maximum(0.0, track_boxes[valid_boxes, 2] - track_boxes[valid_boxes, 0])
            height = np.maximum(0.0, track_boxes[valid_boxes, 3] - track_boxes[valid_boxes, 1])
            area_ratios = width * height

        return {
            "phase": phase,
            "quality": quality,
            "events": events,
            "events_peaks": peaks,
            "events_troughs": troughs,
            "fps": fps,
            "n_frames": len(kps),
            "motion": motion,
            "tracking": {
                "mean_track_score": float(np.nanmean(data.get("track_scores", np.array([np.nan])))),
                "mean_candidate_count": float(np.mean(data.get("candidate_counts", np.array([0])))),
                "visible_box_ratio": float(valid_boxes.mean()) if len(valid_boxes) else 0.0,
                "mean_box_area_px": float(np.mean(area_ratios)) if len(area_ratios) else 0.0,
            },
        }

    def fit_weights(self, features: dict):
        n_ph = len(self.PHASE_CHANNELS)
        n_qu = len(self.QUALITY_CHANNELS)

        if self.cfg.mode == "manual":
            pw = np.array([self.cfg.phase_weights.get(c, 1.0) for c in self.PHASE_CHANNELS], dtype=float)
            qw = np.array([self.cfg.quality_weights.get(c, 1.0) for c in self.QUALITY_CHANNELS], dtype=float)
        else:
            refs = self._reference_names(features)
            if len(refs) >= 2:
                ref_scores = np.array([self.cfg.reference_scores[r] for r in refs], dtype=float)

                qw = []
                for ch in self.QUALITY_CHANNELS:
                    vals = np.array([features[r]["quality"][ch] for r in refs], dtype=float)
                    corr = abs(np.corrcoef(vals, ref_scores)[0, 1]) if np.std(vals) > 1e-8 else 0.0
                    qw.append(max(0.0, corr * np.std(vals)))
                qw = np.asarray(qw, dtype=float)

                pw = []
                for ch in self.PHASE_CHANNELS:
                    vals = np.array([features[r]["phase"][ch].mean() for r in refs], dtype=float)
                    corr = abs(np.corrcoef(vals, ref_scores)[0, 1]) if np.std(vals) > 1e-8 else 0.0
                    pw.append(max(0.0, corr * np.std(vals)))
                pw = np.asarray(pw, dtype=float)
            else:
                qw = np.ones(n_qu)
                pw = np.ones(n_ph)

        if np.allclose(pw.sum(), 0.0):
            pw = np.ones(n_ph)
        if np.allclose(qw.sum(), 0.0):
            qw = np.ones(n_qu)

        self._phase_w = pw / (pw.sum() + 1e-8)
        self._quality_w = qw / (qw.sum() + 1e-8)
        return self._phase_w.copy(), self._quality_w.copy()

    def phase_distance(self, fa: dict, fb: dict) -> float:
        w = self._phase_w if self._phase_w is not None else np.ones(len(self.PHASE_CHANNELS)) / len(self.PHASE_CHANNELS)
        total = 0.0
        for i, ch in enumerate(self.PHASE_CHANNELS):
            a = self._z(fa["phase"][ch]).astype(np.float64)
            b = self._z(fb["phase"][ch]).astype(np.float64)
            total += w[i] * dtw_mod.distance_fast(a, b)
        return float(total)

    def quality_distance(self, fa: dict, fb: dict) -> float:
        w = self._quality_w if self._quality_w is not None else np.ones(len(self.QUALITY_CHANNELS)) / len(self.QUALITY_CHANNELS)
        qa = np.array([fa["quality"][c] for c in self.QUALITY_CHANNELS])
        qb = np.array([fb["quality"][c] for c in self.QUALITY_CHANNELS])
        return float(np.sum(w * np.abs(qa - qb)))

    def mixed_distance(self, fa: dict, fb: dict) -> float:
        return self.cfg.alpha * self.phase_distance(fa, fb) + (1 - self.cfg.alpha) * self.quality_distance(fa, fb)

    def matrix(self, features: dict) -> pd.DataFrame:
        names = list(features.keys())
        n = len(names)
        M = np.zeros((n, n), dtype=float)
        for i in range(n):
            for j in range(i + 1, n):
                d = self.mixed_distance(features[names[i]], features[names[j]])
                M[i, j] = M[j, i] = d
        return pd.DataFrame(M, index=names, columns=names)

    def reference_quality_report(self, features: dict, reference_scores: Optional[Dict[str, float]] = None, temperature: Optional[float] = None) -> pd.DataFrame:
        reference_scores = reference_scores or self.cfg.reference_scores
        refs = [name for name in reference_scores if name in features]
        if len(refs) < 2:
            raise ValueError("Нужно минимум 2 эталонных видео, присутствующих в features.")

        ref_values = np.array([float(reference_scores[name]) for name in refs], dtype=float)
        ref_pairwise = []
        for i in range(len(refs)):
            for j in range(i + 1, len(refs)):
                ref_pairwise.append(self.mixed_distance(features[refs[i]], features[refs[j]]))

        if temperature is None:
            base_scale = float(np.median(ref_pairwise)) if ref_pairwise else 1.0
            temperature = self.cfg.reference_temperature_scale * base_scale
        temperature = max(float(temperature), 1e-6)

        rows = []
        for name in features:
            distances = np.array([self.mixed_distance(features[name], features[ref]) for ref in refs], dtype=float)

            if name in reference_scores:
                ref_weights = np.zeros(len(refs), dtype=float)
                ref_weights[refs.index(name)] = 1.0
                score_01 = float(reference_scores[name])
                closest_reference = name
            else:
                logits = -distances / temperature
                logits -= logits.max()
                ref_weights = np.exp(logits)
                ref_weights /= ref_weights.sum() + 1e-8
                score_01 = float(np.sum(ref_weights * ref_values))
                closest_reference = refs[int(np.argmin(distances))]

            quality_score_100 = 100.0 * score_01
            estimated_level = 1.0 + 3.0 * (1.0 - score_01)
            rows.append({
                "video": name,
                "quality_score_100": quality_score_100,
                "quality_score_01": score_01,
                "estimated_level": estimated_level,
                "closest_reference": closest_reference,
                "temperature": temperature,
                **{f"d_to_{ref}": float(dist) for ref, dist in zip(refs, distances)},
                **{f"ref_weight_{ref}": float(weight) for ref, weight in zip(refs, ref_weights)},
            })

        return pd.DataFrame(rows).sort_values(["quality_score_100", "video"], ascending=[False, True]).reset_index(drop=True)

    def print_quality_ranking(self, features: dict, reference_scores: Optional[Dict[str, float]] = None):
        ranked = self.reference_quality_report(features, reference_scores=reference_scores or self.cfg.reference_scores)
        print("Ranking (reference-calibrated, higher = better):")
        for i, row in ranked.iterrows():
            print(
                f"  {i + 1}. {row['video']:12s} "
                f"score={row['quality_score_100']:6.2f} "
                f"level≈{row['estimated_level']:.2f} "
                f"closest={row['closest_reference']}"
            )
        return ranked

    def quality_breakdown(self, feat: dict, reference_feat: dict) -> pd.DataFrame:
        w = self._quality_w if self._quality_w is not None else np.ones(len(self.QUALITY_CHANNELS)) / len(self.QUALITY_CHANNELS)
        rows = []
        for i, ch in enumerate(self.QUALITY_CHANNELS):
            value = float(feat["quality"][ch])
            ref_value = float(reference_feat["quality"][ch])
            delta = value - ref_value
            rows.append({
                "channel": ch,
                "value": value,
                "reference": ref_value,
                "delta": delta,
                "weighted_gap": abs(delta) * float(w[i]),
            })
        return pd.DataFrame(rows).sort_values("weighted_gap", ascending=False).reset_index(drop=True)

    def render_direction_video(self, name: str, feat: dict, out_name: Optional[str] = None, draw_skeleton: bool = True):
        vpath = self._video_path_for(name)
        if not vpath.exists():
            print(f"Video missing: {vpath}")
            return None

        cap = cv2.VideoCapture(str(vpath))
        if not cap.isOpened():
            print(f"Cannot open: {vpath}")
            return None

        fps = cap.get(cv2.CAP_PROP_FPS) or feat["fps"] or 30.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        out_file = OUTPUT_DIR / (out_name or f"{name}_direction_overlay.mp4")
        writer = cv2.VideoWriter(str(out_file), cv2.VideoWriter_fourcc(*"mp4v"), float(fps), (w, h))

        kps_raw = self._pose_cache[name]["kps"]
        track_boxes = self._pose_cache[name].get("track_boxes", np.full((len(kps_raw), 4), np.nan, dtype=np.float32))
        motion = feat["motion"]
        com = motion["com_px"]
        vel = motion["motion_vec_px"]
        speed = motion["speed_norm"]
        peaks = set(feat.get("events_peaks", np.array([], dtype=int)).tolist())
        troughs = set(feat.get("events_troughs", np.array([], dtype=int)).tolist())

        i = 0
        while True:
            ret, frame = cap.read()
            if not ret or i >= len(kps_raw):
                break

            kp = kps_raw[i]
            box = track_boxes[i]

            if np.isfinite(box).all():
                cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 180, 255), 2)
                cv2.putText(frame, "tracked athlete", (int(box[0]), max(24, int(box[1]) - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (0, 180, 255), 2, cv2.LINE_AA)

            if draw_skeleton:
                for a, b in SKELETON_EDGES:
                    if kp[a, 2] > self.cfg.conf_thr and kp[b, 2] > self.cfg.conf_thr:
                        p1 = (int(kp[a, 0]), int(kp[a, 1]))
                        p2 = (int(kp[b, 0]), int(kp[b, 1]))
                        cv2.line(frame, p1, p2, (70, 255, 150), 2, cv2.LINE_AA)
                m = kp[:, 2] > self.cfg.conf_thr
                for p in kp[m, :2]:
                    cv2.circle(frame, (int(p[0]), int(p[1])), 3, (40, 80, 255), -1, cv2.LINE_AA)

            c = com[i]
            v = vel[i]
            n = np.linalg.norm(v)
            if np.isfinite(c).all() and n > 1e-6:
                u = v / (n + 1e-8)
                start = (int(c[0]), int(c[1] + 30))
                L = int(np.clip(35 + 180 * speed[i], 35, 140))
                end = (int(start[0] + u[0] * L), int(start[1] + u[1] * L))
                cv2.arrowedLine(frame, start, end, (0, 220, 255), 4, cv2.LINE_AA, tipLength=0.30)

            phase_tag = ""
            if i in peaks:
                phase_tag = "PEAK"
            elif i in troughs:
                phase_tag = "TROUGH"
            if phase_tag:
                color = (255, 210, 0) if phase_tag == "PEAK" else (0, 220, 255)
                cv2.putText(frame, phase_tag, (16, 86), cv2.FONT_HERSHEY_SIMPLEX, 0.68, color, 2, cv2.LINE_AA)

            cv2.putText(frame, f"{name} | frame {i + 1}/{total}", (16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"speed_norm={speed[i]:.3f}", (16, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 220, 255), 2, cv2.LINE_AA)
            writer.write(frame)
            i += 1

        cap.release()
        writer.release()
        print(f"  saved: {out_file}")
        return out_file

    def evaluate_external_video(self, video_path, reference_features: dict, video_name: Optional[str] = None, force: bool = False, render_overlay: bool = True) -> dict:
        video_path = pathlib.Path(video_path)
        if not video_path.exists():
            raise FileNotFoundError(f"Video not found: {video_path}")

        safe_name = video_name or re.sub(r"[^0-9a-zA-Z_]+", "_", video_path.stem).strip("_").lower() or "query_video"
        self.run_inference(safe_name, video_path, force=force)
        feat = self.extract(safe_name)
        merged = {**reference_features, safe_name: feat}
        report = self.reference_quality_report(merged, reference_scores=self.cfg.reference_scores)
        row = report[report["video"] == safe_name].reset_index(drop=True)
        closest_ref = row.loc[0, "closest_reference"]
        breakdown = self.quality_breakdown(feat, reference_features[closest_ref])
        overlay_path = None
        if render_overlay:
            overlay_path = self.render_direction_video(safe_name, feat, out_name=f"{safe_name}_evaluation_overlay.mp4")
        return {
            "video_name": safe_name,
            "features": feat,
            "report": row,
            "breakdown": breakdown,
            "overlay_path": overlay_path,
        }


print("Analyzer v4 ready")
print("Phase channels:", len(Analyzer.BASE_PHASE), "(x2 with velocity)")
print("Quality channels:", len(Analyzer.QUALITY_CHANNELS))
print("Removed legacy good_labels/bad_labels: scoring now uses reference_scores only")


In [ ]:
import logging
import warnings
import numpy as np

warnings.filterwarnings(
    "ignore",
    message="The compiled dtaidistance C library is not available.*",
)
logging.getLogger("dtaidistance").setLevel(logging.ERROR)

try:
    _ = dtw_mod.distance_fast(
        np.array([0.0, 1.0], dtype=np.float64),
        np.array([0.0, 1.0], dtype=np.float64),
    )
    _DTW_DISTANCE_FN = dtw_mod.distance_fast
    _DTW_IMPL = "distance_fast"
except Exception:
    _DTW_DISTANCE_FN = dtw_mod.distance
    _DTW_IMPL = "distance"

def _phase_distance_no_spam(self, fa: dict, fb: dict) -> float:
    w = self._phase_w if self._phase_w is not None else np.ones(len(self.PHASE_CHANNELS)) / len(self.PHASE_CHANNELS)
    total = 0.0
    for i, ch in enumerate(self.PHASE_CHANNELS):
        a = self._z(fa["phase"][ch]).astype(np.float64)
        b = self._z(fb["phase"][ch]).astype(np.float64)
        total += w[i] * _DTW_DISTANCE_FN(a, b)
    return float(total)

Analyzer.phase_distance = _phase_distance_no_spam
print(f"DTW anti-spam patch applied (impl={_DTW_IMPL})")


In [ ]:
if "az" not in globals() or "features" not in globals():
    print("Preparing analyzer/features (az, features) ...")
    az = Analyzer(Cfg(**ANALYZER_CONFIG))
    for name, vpath in VIDEOS.items():
        az.run_inference(name, vpath, force=ANALYZER_CONFIG["force_rebuild_reference_cache"])
    features = {name: az.extract(name) for name in VIDEOS}
    az.fit_weights(features)
else:
    print("Using existing az/features from notebook")

print("References ready:", list(features))


In [ ]:
# Multicam low-level fusion core.
# YOLO Pose дает keypoints + confidence для каждой точки.
# Для каждой feature берем confidence только тех keypoints, из которых она считается.
# Дальше одинаковые feature усредняются покадрово между камерами с этими confidence-весами.
# Score считается уже по fused feature, а не по фиксированным весам камер.


DIRECTION_PHASE_CHANNELS = {
    "motion_dir_stability_local", "motion_dir_stability_local_vel",
    "body_motion_align", "body_motion_align_vel",
    "head_motion_align", "head_motion_align_vel",
    "motion_speed_norm", "motion_speed_norm_vel",
}
DIRECTION_QUALITY_CHANNELS = {
    "direction_control",
    "body_motion_alignment",
    "head_motion_alignment",
    "head_body_heading_consistency",
}

FEATURE_KEYPOINTS = {
    "ankles_distance": [KP["l_ankle"], KP["r_ankle"]],
    "knee_angle_mean": [KP["l_hip"], KP["r_hip"], KP["l_knee"], KP["r_knee"], KP["l_ankle"], KP["r_ankle"]],
    "knee_angle_asym": [KP["l_hip"], KP["r_hip"], KP["l_knee"], KP["r_knee"], KP["l_ankle"], KP["r_ankle"]],
    "torso_lean_abs_deg": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "hands_to_torso_mean": [KP["l_wrist"], KP["r_wrist"], KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "head_pitch_rel": [KP["nose"], KP["l_shoulder"], KP["r_shoulder"]],
    "hip_lateral_sway": [KP["l_hip"], KP["r_hip"]],
    "elbow_angle_mean": [KP["l_shoulder"], KP["r_shoulder"], KP["l_elbow"], KP["r_elbow"], KP["l_wrist"], KP["r_wrist"]],
    "shoulder_elevation": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "spine_angle": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "motion_speed_norm": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "motion_dir_stability_local": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "body_motion_align": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "head_motion_align": [KP["nose"], KP["l_shoulder"], KP["r_shoulder"]],
}
QUALITY_KEYPOINTS = {
    "head_up": [KP["nose"], KP["l_shoulder"], KP["r_shoulder"]],
    "torso_upright": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "hands_compact": [KP["l_wrist"], KP["r_wrist"], KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "tempo_stability": [KP["l_ankle"], KP["r_ankle"], KP["l_hip"], KP["r_hip"]],
    "leg_control": [KP["l_hip"], KP["r_hip"], KP["l_knee"], KP["r_knee"], KP["l_ankle"], KP["r_ankle"]],
    "knee_symmetry": [KP["l_hip"], KP["r_hip"], KP["l_knee"], KP["r_knee"], KP["l_ankle"], KP["r_ankle"]],
    "upper_body_stability": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "arm_discipline": [KP["l_shoulder"], KP["r_shoulder"], KP["l_elbow"], KP["r_elbow"], KP["l_wrist"], KP["r_wrist"]],
    "cycle_amplitude_cv": [KP["l_ankle"], KP["r_ankle"]],
    "peak_posture_consistency": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"], KP["l_knee"], KP["r_knee"]],
    "direction_control": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "body_motion_alignment": [KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
    "head_motion_alignment": [KP["nose"], KP["l_shoulder"], KP["r_shoulder"]],
    "head_body_heading_consistency": [KP["nose"], KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]],
}

def _base_channel(ch: str) -> str:
    return ch[:-4] if ch.endswith("_vel") else ch

def _ids_for_channel(ch: str, is_quality: bool = False) -> List[int]:
    if is_quality:
        return QUALITY_KEYPOINTS.get(ch, TORSO_IDS)
    return FEATURE_KEYPOINTS.get(_base_channel(ch), TORSO_IDS)

def _resample_to_n(arr: np.ndarray, n: int) -> np.ndarray:
    if len(arr) == 0:
        return np.zeros(n, dtype=float)
    if len(arr) == n:
        return np.asarray(arr, dtype=float)
    indices = np.linspace(0, len(arr) - 1, n)
    return np.interp(indices, np.arange(len(arr)), np.asarray(arr, dtype=float))

def _sync_slice(feat: dict, spec: dict, sync_start_sec: float, sync_end_sec: float) -> slice:
    fps = float(feat.get("fps", 30.0))
    offset = float(spec.get("offset_sec", 0.0))
    start = max(0, int(round((offset + sync_start_sec) * fps)))
    stop = min(int(feat["n_frames"]), int(round((offset + sync_end_sec) * fps)))
    if stop <= start + 2:
        return slice(0, int(feat["n_frames"]))
    return slice(start, stop)

def _confidence_curve(cam_name: str, feat: dict, spec: dict, ch: str, sync_start_sec: float, sync_end_sec: float, target_n: int, is_quality: bool = False) -> np.ndarray:
    kps = az._pose_cache[cam_name]["kps"]
    conf = np.asarray(kps[:, :, 2], dtype=float)
    ids = _ids_for_channel(ch, is_quality=is_quality)
    sl = _sync_slice(feat, spec, sync_start_sec, sync_end_sec)
    curve = np.nanmean(conf[sl][:, ids], axis=1)
    curve = np.nan_to_num(curve, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(_resample_to_n(curve, target_n), 0.0, 1.0)

def _weighted_mean_stack(stack: np.ndarray, weights: np.ndarray) -> np.ndarray:
    denom = weights.sum(axis=0) + 1e-8
    return (stack * weights).sum(axis=0) / denom

def _weighted_std_stack(stack: np.ndarray, mean: np.ndarray, weights: np.ndarray) -> float:
    denom = weights.sum(axis=0) + 1e-8
    var_t = (((stack - mean[None, :]) ** 2) * weights).sum(axis=0) / denom
    return float(np.mean(np.sqrt(np.maximum(var_t, 0.0))))

def _quality_from_fused_phase(phase: dict, fps: float, fallback_quality: dict) -> tuple:
    dist = np.asarray(phase["ankles_distance"], dtype=float)
    min_dist = max(2, int(fps * 0.3))
    peaks, _ = find_peaks(dist, height=np.percentile(dist, 40), distance=min_dist)
    troughs, _ = find_peaks(-dist, height=np.percentile(-dist, 40), distance=min_dist)
    events = np.sort(np.unique(np.concatenate([peaks, troughs])))
    iei = np.diff(events) / fps if len(events) > 1 else np.array([1.0])
    peak_ankl = dist[events] if len(events) > 0 else dist
    peak_torso = np.asarray(phase["torso_lean_abs_deg"])[events] if len(events) > 0 else np.asarray(phase["torso_lean_abs_deg"])
    ankl_mean = dist.mean() + 1e-8
    speed = np.asarray(phase["motion_speed_norm"], dtype=float)
    rel_mask = speed > np.percentile(speed, 35)
    if rel_mask.sum() < 5:
        rel_mask = np.ones_like(speed, dtype=bool)
    quality = {
        "head_up": -float(np.median(phase["head_pitch_rel"])),
        "torso_upright": -float(np.median(phase["torso_lean_abs_deg"])),
        "hands_compact": -float(np.median(phase["hands_to_torso_mean"])),
        "tempo_stability": -float(iei.std() / (iei.mean() + 1e-8)),
        "leg_control": -float(np.asarray(phase["ankles_distance"]).std() / ankl_mean),
        "knee_symmetry": -float(np.median(phase["knee_angle_asym"])),
        "upper_body_stability": -float(np.asarray(phase["torso_lean_abs_deg"]).std()),
        "arm_discipline": -float(np.asarray(phase["hands_to_torso_mean"]).std()),
        "cycle_amplitude_cv": -float(peak_ankl.std() / (peak_ankl.mean() + 1e-8)),
        "peak_posture_consistency": -float(peak_torso.std()),
        "direction_control": float(np.median(np.asarray(phase["motion_dir_stability_local"])[rel_mask])),
        "body_motion_alignment": float(np.median(np.asarray(phase["body_motion_align"])[rel_mask])),
        "head_motion_alignment": float(np.median(np.asarray(phase["head_motion_align"])[rel_mask])),
        "head_body_heading_consistency": float(fallback_quality["head_body_heading_consistency"]),
    }
    return quality, events, peaks, troughs

def build_multicam_neutral_feature(group_name: str, group_spec: dict, force: bool = False, render_overlays: bool = False) -> dict:
    # Для каждой камеры поднимаем pose и считаем обычные single-camera фичи.
    # Дальше из них уже собираем один общий fused feature dict.
    camera_features = {}
    camera_meta = []
    for spec in group_spec["cameras"]:
        path = pathlib.Path(spec["path"])
        cam_name = f"{group_name}_{spec['name']}"
        print(f"Processing {cam_name}...")
        az.run_inference(cam_name, path, force=force)
        feat = az.extract(cam_name)
        camera_features[cam_name] = feat
        if render_overlays:
            az.render_direction_video(cam_name, feat, out_name=f"{cam_name}_camera_overlay.mp4")
        all_conf = np.asarray(az._pose_cache[cam_name]["kps"][:, :, 2], dtype=float)
        camera_meta.append({
            "name": cam_name,
            "spec": spec,
            "frames": int(feat["n_frames"]),
            "events": int(len(feat.get("events", []))),
            "track_score": float(feat["tracking"]["mean_track_score"]),
            "keypoint_conf_mean": float(np.nanmean(all_conf)),
            "keypoint_conf_p25": float(np.nanpercentile(all_conf, 25)),
        })

    # Определяем общее синхронное окно, внутри которого камеры считаются одной сценой.
    sync_start = float(group_spec.get("sync_start_sec", 0.0))
    sync_end = group_spec.get("sync_end_sec")
    if sync_end is None:
        durations = []
        for meta in camera_meta:
            feat = camera_features[meta["name"]]
            offset = float(meta["spec"].get("offset_sec", 0.0))
            durations.append(float(feat["n_frames"]) / float(feat["fps"]) - offset)
        sync_end = min(durations)
    sync_end = float(sync_end)

    # Приводим все кривые к одной длине, чтобы усреднять их кадр-в-кадр.
    target_n = int(round(np.median([(sync_end - sync_start) * float(camera_features[m["name"]]["fps"]) for m in camera_meta])))
    target_n = max(8, target_n)

    # Phase-каналы объединяем по confidence нужных keypoints.
    # Например, фичи ног больше опираются на камеру, где лучше видны ankles/knees.
    phase = {}
    phase_std = {}
    phase_conf_mean = {}
    for ch in az.PHASE_CHANNELS:
        curves = []
        confs = []
        for meta in camera_meta:
            feat = camera_features[meta["name"]]
            spec = meta["spec"]
            curves.append(_resample_to_n(np.asarray(feat["phase"][ch])[_sync_slice(feat, spec, sync_start, sync_end)], target_n))
            confs.append(_confidence_curve(meta["name"], feat, spec, ch, sync_start, sync_end, target_n, is_quality=False))
        stack = np.vstack(curves)
        weights = np.vstack(confs)
        if float(weights.sum()) <= 1e-8:
            weights = np.ones_like(stack)
        phase[ch] = _weighted_mean_stack(stack, weights)
        phase_std[ch] = _weighted_std_stack(stack, phase[ch], weights)
        phase_conf_mean[ch] = dict(zip([m["name"] for m in camera_meta], np.mean(weights, axis=1).tolist()))

    # Для scalar quality агрегируем значения мягче: через confidence-weighted average.
    # Это нужно как fallback там, где quality нельзя честно пересчитать прямо из fused-кривых.
    fallback_quality = {}
    quality_std = {}
    quality_conf = {}
    for ch in az.QUALITY_CHANNELS:
        vals = np.asarray([float(camera_features[m["name"]]["quality"][ch]) for m in camera_meta], dtype=float)
        conf = np.asarray([
            float(np.percentile(_confidence_curve(m["name"], camera_features[m["name"]], m["spec"], ch, sync_start, sync_end, target_n, is_quality=True), 25))
            for m in camera_meta
        ], dtype=float)
        if float(conf.sum()) <= 1e-8:
            conf = np.ones_like(vals)
        fallback_quality[ch] = float(np.average(vals, weights=conf))
        quality_std[ch] = float(np.sqrt(np.average((vals - fallback_quality[ch]) ** 2, weights=conf)))
        quality_conf[ch] = dict(zip([m["name"] for m in camera_meta], conf.tolist()))

    # После fusion phase-кривых пересчитываем quality уже по fused-сигналу.
    # Так порядок честный: сначала объединяем камеры, потом считаем итоговые метрики.
    quality, events, peaks, troughs = _quality_from_fused_phase(
        phase=phase,
        fps=float(np.median([float(camera_features[m["name"]]["fps"]) for m in camera_meta])),
        fallback_quality=fallback_quality,
    )
    first = next(iter(camera_features.values()))
    # Возвращаем объект того же формата, что и обычный feature dict,
    # чтобы его можно было дальше скормить в тот же reference-based scoring pipeline.
    return {
        **{k: v for k, v in first.items() if k not in {"phase", "quality", "events", "events_peaks", "events_troughs"}},
        "n_frames": target_n,
        "fps": float(np.median([float(camera_features[m["name"]]["fps"]) for m in camera_meta])),
        "phase": phase,
        "quality": quality,
        "events": events,
        "events_peaks": peaks,
        "events_troughs": troughs,
        "tracking": {
            "mean_track_score": float(np.mean([m["track_score"] for m in camera_meta])),
            "mean_candidate_count": float(np.mean([camera_features[m["name"]]["tracking"]["mean_candidate_count"] for m in camera_meta])),
            "camera_count": len(camera_meta),
        },
        "multicam_neutral": {
            "camera_meta": camera_meta,
            "fusion_weighting": "framewise_per_feature_keypoint_confidence",
            "sync_start_sec": sync_start,
            "sync_end_sec": sync_end,
            "phase_std": phase_std,
            "quality_std": quality_std,
            "phase_conf_mean": phase_conf_mean,
            "quality_conf_p25": quality_conf,
            "fallback_quality_conf_weighted": fallback_quality,
            "quality_source": "recomputed_from_fused_phase_where_possible",
            "direction_phase_std_mean": float(np.mean([phase_std[ch] for ch in DIRECTION_PHASE_CHANNELS if ch in phase_std])),
            "direction_quality_std_mean": float(np.mean([quality_std[ch] for ch in DIRECTION_QUALITY_CHANNELS if ch in quality_std])),
        },
    }

def evaluate_multicam_neutral_groups(groups: Dict[str, dict], force: bool = False, render_overlays: bool = False) -> dict:
    # Это тонкая обёртка над fusion-функцией: прогоняет группы камер и печатает краткую сводку.
    rows = {}
    for group_name, spec in groups.items():
        print(f"\n[multicam fusion] {group_name}")
        feat = build_multicam_neutral_feature(group_name, spec, force=force, render_overlays=render_overlays)
        merged = {**features, group_name: feat}
        report = az.reference_quality_report(merged, reference_scores=az.cfg.reference_scores)
        row = report[report["video"] == group_name].reset_index(drop=True).iloc[0]
        print(f"  quality_score_100: {float(row['quality_score_100']):.2f}")
        print(f"  closest_reference: {row['closest_reference']}")
        print(f"  track_score: {feat['tracking']['mean_track_score']:.3f}")
        print(f"  events: {len(feat['events'])}")
        rows[group_name] = {
            "group": group_name,
            "camera_count": feat["tracking"]["camera_count"],
            "quality_score_100": float(row["quality_score_100"]),
            "closest_reference": row["closest_reference"],
            "track_score": feat["tracking"]["mean_track_score"],
            "events": len(feat["events"]),
            "direction_phase_std_mean": feat["multicam_neutral"]["direction_phase_std_mean"],
            "direction_quality_std_mean": feat["multicam_neutral"]["direction_quality_std_mean"],
        }
    return rows


## Helpers and final run configs


In [ ]:
# Final pose-only exercise multicam helpers.
# Чтобы прогнать другой набор, поменяй только MULTICAM_RUN_CONFIG ниже.
# Чтобы прогнать другой набор, поменяй только MULTICAM_RUN_CONFIG ниже.

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import shutil

try:
    _IN_NOTEBOOK_OUTPUT = get_ipython().__class__.__name__ == "ZMQInteractiveShell"
except Exception:
    _IN_NOTEBOOK_OUTPUT = False
if not _IN_NOTEBOOK_OUTPUT:
    plt.switch_backend("Agg")

_REQUIRED = ["az", "features", "build_multicam_neutral_feature", "_confidence_curve", "_resample_to_n"]
_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise RuntimeError(f"Сначала выполни верхние ячейки ноутбука, не хватает: {_missing}")

SYNCED_CUTS_ROOT = DATA_ROOT / "synced_cuts"
OSMISLYNIE_ROOT = OUTPUT_DIR / "multicam_runs"
OSMISLYNIE_ROOT.mkdir(parents=True, exist_ok=True)


def _show_fig():
    if _IN_NOTEBOOK_OUTPUT:
        plt.show()
    else:
        plt.close()


def _safe_name(text: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "_-" else "_" for ch in text).strip("_")


def _find_video(filename: str, source_dir: pathlib.Path) -> pathlib.Path:
    direct = pathlib.Path(source_dir) / filename
    if direct.exists():
        return direct
    root_direct = ROOT / filename
    if root_direct.exists():
        return root_direct
    for search_root in [DATA_ROOT, ROOT]:
        if pathlib.Path(search_root).exists():
            matches = sorted(pathlib.Path(search_root).rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Не нашел {filename}. Положи видео в {source_dir}")


def _cut_interval(src: pathlib.Path, dst: pathlib.Path, start_sec: float, end_sec: float) -> dict:
    # Интервал режется как [start_sec, end_sec), то есть конец не включается.
    cap = cv2.VideoCapture(str(src))
    if not cap.isOpened():
        raise RuntimeError(f"Не могу открыть видео: {src}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start_frame = max(0, int(round(float(start_sec) * fps)))
    end_frame = min(total, int(round(float(end_sec) * fps)))
    if end_frame <= start_frame + 2:
        cap.release()
        raise ValueError(f"Слишком короткая нарезка: {src}, {start_sec}-{end_sec}s")
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    writer = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*"mp4v"), float(fps), (width, height))
    written = 0
    for _ in range(start_frame, end_frame):
        ok, frame = cap.read()
        if not ok:
            break
        writer.write(frame)
        written += 1
    cap.release()
    writer.release()
    return {
        "fps": float(fps),
        "frames": int(written),
        "duration": float(written / fps),
        "size": f"{width}x{height}",
        "source_frames": f"{start_frame}-{start_frame + written - 1}",
    }


def _draw_pose(frame, video_name: str, frame_idx: int):
    out = frame.copy()
    cache = az._pose_cache[video_name]
    kps = cache["kps"]
    frame_idx = min(max(0, int(frame_idx)), len(kps) - 1)
    kp = kps[frame_idx]
    boxes = cache.get("track_boxes")
    if boxes is not None and frame_idx < len(boxes):
        box = boxes[frame_idx]
        if np.all(np.isfinite(box)) and (box[2] - box[0]) > 2 and (box[3] - box[1]) > 2:
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(out, (x1, y1), (x2, y2), (255, 255, 0), 2)
    for a, b in SKELETON_EDGES:
        if kp[a, 2] >= az.cfg.conf_thr and kp[b, 2] >= az.cfg.conf_thr:
            ax, ay = int(kp[a, 0]), int(kp[a, 1])
            bx, by = int(kp[b, 0]), int(kp[b, 1])
            cv2.line(out, (ax, ay), (bx, by), (0, 255, 0), 2)
    for x, y, conf in kp:
        if conf >= az.cfg.conf_thr:
            cv2.circle(out, (int(x), int(y)), 4, (0, 0, 255), -1)
    cv2.putText(out, f"frame {frame_idx}", (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    return out


def _render_pose_overlay_video(video_name: str, video_path: pathlib.Path, out_path: pathlib.Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Не могу открыть нарезанный клип: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or float(az._pose_cache[video_name].get("fps", 30.0))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), float(fps), (width, height))
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok or idx >= len(az._pose_cache[video_name]["kps"]):
            break
        writer.write(_draw_pose(frame, video_name, idx))
        idx += 1
    cap.release()
    writer.release()
    return out_path


def _plot_scores(score_df: pd.DataFrame, graph_dir: pathlib.Path):
    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ["#78a8d8" if kind == "single_camera" else "#222222" for kind in score_df["kind"]]
    ax.bar(score_df["label"], score_df["score"].astype(float), color=colors)
    ax.set_ylim(0, 100)
    ax.set_ylabel("reference similarity score")
    ax.set_title("Single cameras vs 3cam fused score")
    for i, row in score_df.iterrows():
        ax.text(i, float(row["score"]) + 1.2, f"{row['score']:.1f}\n{row['closest_reference']}", ha="center", va="bottom", fontsize=9)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(graph_dir / "scores.png", dpi=160, bbox_inches="tight")
    _show_fig()



MULTICAM_RUN_CONFIG = {
    "run_name": "run_IMG_6449_13666886683285_20260414_122625_sync_cut",
    "source_dir": SYNCED_CUTS_ROOT / "run_IMG_6449_13666886683285_20260414_122625_sync_cut",
    "group_name": "osmislynie2_IMG_6449_13666886683285_20260414_122625_sync_cut",
    "common_sync_sec": 4.4,
    "force_rebuild_pose": False,
    "run_config": [
        {"label": "IMG_6449", "name": "cam1_IMG_6449", "file": "IMG_6449_cut_1_5.4s.mp4", "start_sec": 0.0, "end_sec": 4.4},
        {"label": "13666886683285", "name": "cam2_13666886683285", "file": "13666886683285_cut_2.37_6.77s.mp4", "start_sec": 0.0, "end_sec": 4.4},
        {"label": "20260414_122625", "name": "cam3_20260414_122625", "file": "20260414_122625_cut_9.6_14s.mp4", "start_sec": 0.0, "end_sec": 4.4},
    ],
}


## Confidence/observability fusion


In [ ]:
V4_MIN_OBSERVABILITY = 0.08


def _v4_clip01(x):
    return np.clip(np.asarray(x, dtype=float), 0.0, 1.0)


def _v4_ramp(x, lo: float, hi: float):
    x = np.asarray(x, dtype=float)
    return _v4_clip01((x - lo) / (hi - lo + 1e-8))


def _v4_dist(kps: np.ndarray, a: int, b: int) -> np.ndarray:
    return np.linalg.norm(kps[:, a, :2] - kps[:, b, :2], axis=1)


def _v4_mid(kps: np.ndarray, a: int, b: int) -> np.ndarray:
    return 0.5 * (kps[:, a, :2] + kps[:, b, :2])


def _v4_body_scale(kps: np.ndarray) -> np.ndarray:
    shoulder_mid = _v4_mid(kps, KP["l_shoulder"], KP["r_shoulder"])
    hip_mid = _v4_mid(kps, KP["l_hip"], KP["r_hip"])
    torso = np.linalg.norm(shoulder_mid - hip_mid, axis=1)
    shoulder_w = _v4_dist(kps, KP["l_shoulder"], KP["r_shoulder"])
    hip_w = _v4_dist(kps, KP["l_hip"], KP["r_hip"])
    leg_l = _v4_dist(kps, KP["l_hip"], KP["l_knee"]) + _v4_dist(kps, KP["l_knee"], KP["l_ankle"])
    leg_r = _v4_dist(kps, KP["r_hip"], KP["r_knee"]) + _v4_dist(kps, KP["r_knee"], KP["r_ankle"])
    scale = np.nanmedian(np.vstack([torso * 1.8, shoulder_w * 2.8, hip_w * 3.2, leg_l * 0.75, leg_r * 0.75]), axis=0)
    finite_positive = scale[np.isfinite(scale) & (scale > 1.0)]
    fallback = float(np.nanmedian(finite_positive)) if len(finite_positive) else 100.0
    scale = np.nan_to_num(scale, nan=fallback, posinf=fallback, neginf=fallback)
    return np.maximum(scale, 1.0)


def _v4_segment_score(kps: np.ndarray, pairs: list, scale: np.ndarray) -> np.ndarray:
    vals = []
    for a, b in pairs:
        vals.append(_v4_ramp(_v4_dist(kps, a, b) / scale, 0.05, 0.35))
    if not vals:
        return np.ones(len(kps), dtype=float)
    return np.nanmean(np.vstack(vals), axis=0)


def _v4_torso_center(kps: np.ndarray) -> np.ndarray:
    return 0.5 * (_v4_mid(kps, KP["l_shoulder"], KP["r_shoulder"]) + _v4_mid(kps, KP["l_hip"], KP["r_hip"]))


def _v4_motion_score(kps: np.ndarray, scale: np.ndarray) -> np.ndarray:
    center = _v4_torso_center(kps)
    speed = np.linalg.norm(np.diff(center, axis=0, prepend=center[:1]), axis=1) / (scale + 1e-8)
    return 0.35 + 0.65 * _v4_ramp(speed, 0.005, 0.08)


def _v4_observability_for_channel(kps: np.ndarray, ch: str) -> np.ndarray:
    base = _base_channel(ch)
    scale = _v4_body_scale(kps)
    n = len(kps)

    leg_channels = {
        "ankles_distance", "knee_angle_mean", "knee_angle_asym", "hip_lateral_sway",
        "cycle_amplitude_cv", "leg_control", "knee_symmetry", "tempo_stability",
    }
    torso_channels = {
        "torso_lean_abs_deg", "spine_angle", "shoulder_elevation", "torso_upright",
        "upper_body_stability", "peak_posture_consistency",
    }
    arm_channels = {
        "hands_to_torso_mean", "elbow_angle_mean", "hands_compact", "arm_discipline",
    }
    head_channels = {
        "head_pitch_rel", "head_up", "head_motion_align", "head_motion_alignment",
        "head_body_heading_consistency",
    }
    motion_channels = {
        "motion_speed_norm", "motion_dir_stability_local", "body_motion_align",
        "direction_control", "body_motion_alignment",
    }

    if base in leg_channels:
        leg_pairs = [
            (KP["l_hip"], KP["l_knee"]), (KP["l_knee"], KP["l_ankle"]),
            (KP["r_hip"], KP["r_knee"]), (KP["r_knee"], KP["r_ankle"]),
        ]
        segment = _v4_segment_score(kps, leg_pairs, scale)
        ankle_sep = _v4_dist(kps, KP["l_ankle"], KP["r_ankle"]) / (scale + 1e-8)
        knee_sep = _v4_dist(kps, KP["l_knee"], KP["r_knee"]) / (scale + 1e-8)
        # Separation is a soft cue only: it helps avoid collapsed side projections,
        # but it never fully removes a camera because the athlete can legitimately bring feet closer.
        separation = 0.45 + 0.55 * np.maximum(_v4_ramp(ankle_sep, 0.04, 0.45), _v4_ramp(knee_sep, 0.04, 0.32))
        obs = segment * separation
    elif base in torso_channels:
        shoulder_mid = _v4_mid(kps, KP["l_shoulder"], KP["r_shoulder"])
        hip_mid = _v4_mid(kps, KP["l_hip"], KP["r_hip"])
        torso_len = np.linalg.norm(shoulder_mid - hip_mid, axis=1) / (scale + 1e-8)
        shoulder_w = _v4_dist(kps, KP["l_shoulder"], KP["r_shoulder"]) / (scale + 1e-8)
        hip_w = _v4_dist(kps, KP["l_hip"], KP["r_hip"]) / (scale + 1e-8)
        obs = 0.50 * _v4_ramp(torso_len, 0.10, 0.45) + 0.25 * _v4_ramp(shoulder_w, 0.05, 0.32) + 0.25 * _v4_ramp(hip_w, 0.04, 0.28)
    elif base in arm_channels:
        arm_pairs = [
            (KP["l_shoulder"], KP["l_elbow"]), (KP["l_elbow"], KP["l_wrist"]),
            (KP["r_shoulder"], KP["r_elbow"]), (KP["r_elbow"], KP["r_wrist"]),
        ]
        obs = _v4_segment_score(kps, arm_pairs, scale)
    elif base in head_channels:
        shoulder_mid = _v4_mid(kps, KP["l_shoulder"], KP["r_shoulder"])
        head_dist = np.linalg.norm(kps[:, KP["nose"], :2] - shoulder_mid, axis=1) / (scale + 1e-8)
        obs = _v4_ramp(head_dist, 0.04, 0.28)
    elif base in motion_channels:
        torso_pairs = [(KP["l_shoulder"], KP["r_shoulder"]), (KP["l_hip"], KP["r_hip"])]
        obs = _v4_segment_score(kps, torso_pairs, scale) * _v4_motion_score(kps, scale)
    else:
        # Fallback: if the feature is unknown, use a torso geometry score.
        torso_pairs = [(KP["l_shoulder"], KP["r_shoulder"]), (KP["l_hip"], KP["r_hip"])]
        obs = _v4_segment_score(kps, torso_pairs, scale)

    obs = np.nan_to_num(obs, nan=V4_MIN_OBSERVABILITY, posinf=1.0, neginf=V4_MIN_OBSERVABILITY)
    return np.clip(obs, V4_MIN_OBSERVABILITY, 1.0)


def _v4_observability_curve(cam_name: str, feat: dict, spec: dict, ch: str, sync_start: float, sync_end: float, target_n: int) -> np.ndarray:
    kps = np.asarray(az._pose_cache[cam_name]["kps"], dtype=float)
    obs = _v4_observability_for_channel(kps, ch)
    sl = _sync_slice(feat, spec, sync_start, sync_end)
    return np.clip(_resample_to_n(obs[sl], target_n), V4_MIN_OBSERVABILITY, 1.0)


def _v4_weight_curve(cam_name: str, feat: dict, spec: dict, ch: str, sync_start: float, sync_end: float, target_n: int, is_quality: bool = False) -> tuple:
    conf = _confidence_curve(cam_name, feat, spec, ch, sync_start, sync_end, target_n, is_quality=is_quality)
    obs = _v4_observability_curve(cam_name, feat, spec, ch, sync_start, sync_end, target_n)
    return conf, obs, conf * obs


MULTICAM_6448_CONFIG = {
    "run_name": "run_13666905229973_20260414_122442_IMG_6448_first4s",
    "source_dir": SYNCED_CUTS_ROOT / "run_13666905229973_20260414_122442_IMG_6448_first4s",
    "group_name": "osmislynie2_13666905229973_20260414_122442_IMG_6448_first4s",
    "common_sync_sec": 4.0,
    "force_rebuild_pose": False,
    "run_config": [
        {"label": "13666905229973", "name": "cam1_13666905229973", "file": "13666905229973_cut_1_5s.mp4", "start_sec": 0.0, "end_sec": 4.0},
        {"label": "20260414_122442", "name": "cam2_20260414_122442", "file": "20260414_122442_cut_0_4s.mp4", "start_sec": 0.0, "end_sec": 4.0},
        {"label": "IMG_6448", "name": "cam3_IMG_6448", "file": "IMG_6448_cut_0_4s.mp4", "start_sec": 0.0, "end_sec": 4.0},
    ],
}


In [ ]:
V41_CONF_THRESHOLD = 0.50


def _v41_threshold_weights_with_fallback(raw_weights: np.ndarray, confs: np.ndarray, conf_threshold: float = V41_CONF_THRESHOLD) -> np.ndarray:
    weights = np.where(confs >= conf_threshold, raw_weights, 0.0)
    empty_frames = weights.sum(axis=0) <= 1e-8
    if np.any(empty_frames):
        weights[:, empty_frames] = raw_weights[:, empty_frames]
    return weights


def _plot_feature_fusion_v41(plot_cams, single_feats, fused_feat, common_sync_sec: float, graph_dir: pathlib.Path, conf_threshold: float = V41_CONF_THRESHOLD):
    n_plot = int(fused_feat["n_frames"])
    fps_plot = float(fused_feat["fps"])
    time = np.arange(n_plot) / fps_plot

    def phase(video_name: str, ch: str):
        return _resample_to_n(np.asarray(single_feats[video_name]["phase"][ch], dtype=float), n_plot)

    def weight(cam, ch: str):
        video_name = cam["video_name"]
        conf, _, raw = _v4_weight_curve(video_name, single_feats[video_name], {"offset_sec": 0.0}, ch, 0.0, common_sync_sec, n_plot)
        return np.where(conf >= conf_threshold, raw, 0.0), raw

    specs = [("ankles_distance", "legs: ankles_distance"), ("torso_lean_abs_deg", "torso: torso_lean_abs_deg")]

    fig, axes = plt.subplots(len(specs), 2, figsize=(13, 3.35 * len(specs)), sharex=True)
    if len(specs) == 1:
        axes = np.asarray([axes])

    for row_idx, (ch, title) in enumerate(specs):
        gated_weights = []
        raw_weights = []
        for cam in plot_cams:
            axes[row_idx, 0].plot(time, phase(cam["video_name"], ch), lw=1.4, alpha=0.78, label=cam["label"])
            gated, raw = weight(cam, ch)
            gated_weights.append(gated)
            raw_weights.append(raw)
        gated_weights = np.vstack(gated_weights)
        raw_weights = np.vstack(raw_weights)
        empty_frames = gated_weights.sum(axis=0) <= 1e-8
        if np.any(empty_frames):
            gated_weights[:, empty_frames] = raw_weights[:, empty_frames]
        relative_weights = gated_weights / (gated_weights.sum(axis=0, keepdims=True) + 1e-8)
        for cam, contribution in zip(plot_cams, relative_weights):
            axes[row_idx, 1].plot(time, contribution, lw=1.4, alpha=0.86, label=cam["label"])
        axes[row_idx, 0].plot(
            time,
            _resample_to_n(np.asarray(fused_feat["phase"][ch], dtype=float), n_plot),
            color="black",
            lw=2.6,
            label="3cam fused",
        )
        axes[row_idx, 0].set_title(f"{title}: values")
        axes[row_idx, 1].set_title(f"{title}: contribution after confidence threshold")
        axes[row_idx, 0].grid(alpha=0.25)
        axes[row_idx, 1].grid(alpha=0.25)
        axes[row_idx, 1].set_ylim(-0.05, 1.05)
        axes[row_idx, 0].legend(fontsize=8)
        axes[row_idx, 1].legend(fontsize=8)

    axes[-1, 0].set_xlabel("synced time, sec")
    axes[-1, 1].set_xlabel("synced time, sec")
    fig.suptitle("Feature-level confidence fusion", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(graph_dir / "feature_fusion_v41_auto_observability_threshold.png", dpi=160, bbox_inches="tight")
    _show_fig()


In [ ]:
# Safety patch for confidence-weighted fusion.
# If every camera has zero weight in a frame, avoid artificial fused_feature=0.
# In that rare case, use equal weights for that frame.

_BASE_V41_THRESHOLD_WEIGHTS_WITH_FALLBACK = globals().get(
    "_BASE_V41_THRESHOLD_WEIGHTS_WITH_FALLBACK", _v41_threshold_weights_with_fallback
)
_BASE_WEIGHTED_MEAN_STACK = globals().get("_BASE_WEIGHTED_MEAN_STACK", _weighted_mean_stack)
_BASE_WEIGHTED_STD_STACK = globals().get("_BASE_WEIGHTED_STD_STACK", _weighted_std_stack)


def _v41_threshold_weights_with_fallback(raw_weights: np.ndarray, confs: np.ndarray, conf_threshold: float = V41_CONF_THRESHOLD) -> np.ndarray:
    raw_weights = np.asarray(raw_weights, dtype=float)
    confs = np.asarray(confs, dtype=float)
    weights = np.where(confs >= conf_threshold, raw_weights, 0.0)

    empty_frames = weights.sum(axis=0) <= 1e-8
    if np.any(empty_frames):
        raw_sum = raw_weights.sum(axis=0)
        raw_ok = empty_frames & (raw_sum > 1e-8)
        if np.any(raw_ok):
            weights[:, raw_ok] = raw_weights[:, raw_ok]

        still_empty = weights.sum(axis=0) <= 1e-8
        if np.any(still_empty):
            weights[:, still_empty] = 1.0
    return weights


def _weighted_mean_stack(stack: np.ndarray, weights: np.ndarray) -> np.ndarray:
    stack = np.asarray(stack, dtype=float)
    weights = np.asarray(weights, dtype=float)
    denom = weights.sum(axis=0)
    safe_weights = weights.copy()
    empty_frames = denom <= 1e-8
    if np.any(empty_frames):
        safe_weights[:, empty_frames] = 1.0
        denom = safe_weights.sum(axis=0)
    return (stack * safe_weights).sum(axis=0) / (denom + 1e-8)


def _weighted_std_stack(stack: np.ndarray, mean: np.ndarray, weights: np.ndarray) -> float:
    stack = np.asarray(stack, dtype=float)
    mean = np.asarray(mean, dtype=float)
    weights = np.asarray(weights, dtype=float)
    denom = weights.sum(axis=0)
    safe_weights = weights.copy()
    empty_frames = denom <= 1e-8
    if np.any(empty_frames):
        safe_weights[:, empty_frames] = 1.0
        denom = safe_weights.sum(axis=0)
    var_t = (((stack - mean[None, :]) ** 2) * safe_weights).sum(axis=0) / (denom + 1e-8)
    return float(np.mean(np.sqrt(np.maximum(var_t, 0.0))))


def _with_suffix_config(config: dict, suffix: str) -> dict:
    cfg = dict(config)
    cfg["run_name"] = f"{cfg['run_name']}_{suffix}"
    cfg["group_name"] = f"{cfg['group_name']}_{suffix}"
    return cfg


## Active-athlete ROI and final pose-only exercise experiment


In [ ]:
V42_USE_CONDITIONAL_ROI = True


V42_AUTO_ROI_MARGIN = 0.45


V42_AUTO_ROI_MIN_VALID_BOX_RATIO = 0.20


V42_AUTO_ROI_MIN_CROP_FRAC = 0.45


V42_AUTO_ROI_MEAN_CONF_THR = 0.55


V42_AUTO_ROI_VISIBLE_RATIO_THR = 0.85


V42_AUTO_ROI_SMALL_BBOX_THR = 0.025


V42_AUTO_ROI_SMALL_BBOX_CONF_THR = 0.75


V42_AUTO_ROI_MIN_ZOOM_FACTOR = 1.15


def _v42_video_hw(path: pathlib.Path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return 1, 1, 30.0, 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 1)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return max(width, 1), max(height, 1), float(fps), int(frames)


def _v42_expand_roi_keep_aspect(x1, y1, x2, y2, width, height, margin=V42_AUTO_ROI_MARGIN, min_crop_frac=V42_AUTO_ROI_MIN_CROP_FRAC):
    frame_aspect = width / max(height, 1)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    bw = max(2.0, x2 - x1)
    bh = max(2.0, y2 - y1)
    crop_w = max(bw * (1.0 + 2.0 * margin), width * min_crop_frac)
    crop_h = max(bh * (1.0 + 2.0 * margin), height * min_crop_frac)
    if crop_w / max(crop_h, 1e-6) > frame_aspect:
        crop_h = crop_w / frame_aspect
    else:
        crop_w = crop_h * frame_aspect
    crop_w = min(crop_w, width)
    crop_h = min(crop_h, height)
    x1 = cx - crop_w / 2.0
    x2 = cx + crop_w / 2.0
    y1 = cy - crop_h / 2.0
    y2 = cy + crop_h / 2.0
    if x1 < 0:
        x2 -= x1
        x1 = 0
    if y1 < 0:
        y2 -= y1
        y1 = 0
    if x2 > width:
        x1 -= x2 - width
        x2 = width
    if y2 > height:
        y1 -= y2 - height
        y2 = height
    x1 = max(0.0, min(float(width - 2), x1))
    y1 = max(0.0, min(float(height - 2), y1))
    x2 = max(x1 + 2.0, min(float(width), x2))
    y2 = max(y1 + 2.0, min(float(height), y2))
    return x1, y1, x2, y2


def _v42_compute_auto_roi_from_cache(video_name: str, video_path: pathlib.Path, margin=V42_AUTO_ROI_MARGIN, min_valid_ratio=V42_AUTO_ROI_MIN_VALID_BOX_RATIO):
    width, height, fps, frame_count = _v42_video_hw(video_path)
    cache = az._pose_cache.get(video_name, {})
    boxes = np.asarray(cache.get("track_boxes", np.empty((0, 4))), dtype=float)
    if len(boxes) == 0:
        return {
            "roi_px": (0, 0, width, height),
            "roi_norm": (0.0, 0.0, 1.0, 1.0),
            "valid_box_ratio": 0.0,
            "zoom_factor": 1.0,
            "reason": "no_track_boxes",
        }
    valid = np.isfinite(boxes).all(axis=1) & ((boxes[:, 2] - boxes[:, 0]) > 4) & ((boxes[:, 3] - boxes[:, 1]) > 4)
    valid_ratio = float(valid.mean()) if len(valid) else 0.0
    if valid_ratio < min_valid_ratio:
        return {
            "roi_px": (0, 0, width, height),
            "roi_norm": (0.0, 0.0, 1.0, 1.0),
            "valid_box_ratio": valid_ratio,
            "zoom_factor": 1.0,
            "reason": "too_few_valid_boxes",
        }
    vb = boxes[valid]
    x1 = float(np.percentile(vb[:, 0], 5))
    y1 = float(np.percentile(vb[:, 1], 5))
    x2 = float(np.percentile(vb[:, 2], 95))
    y2 = float(np.percentile(vb[:, 3], 95))
    x1, y1, x2, y2 = _v42_expand_roi_keep_aspect(x1, y1, x2, y2, width, height, margin=margin)
    roi_px = (int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2)))
    crop_w = max(1, roi_px[2] - roi_px[0])
    crop_h = max(1, roi_px[3] - roi_px[1])
    zoom_factor = float(min(width / crop_w, height / crop_h))
    return {
        "roi_px": roi_px,
        "roi_norm": (roi_px[0] / width, roi_px[1] / height, roi_px[2] / width, roi_px[3] / height),
        "valid_box_ratio": valid_ratio,
        "zoom_factor": zoom_factor,
        "reason": "auto_roi_from_track_boxes",
    }


def _v42_first_pass_diagnostics(video_name: str, video_path: pathlib.Path) -> dict:
    width, height, _, _ = _v42_video_hw(video_path)
    cache = az._pose_cache.get(video_name, {})
    kps = np.asarray(cache.get("kps", np.empty((0, len(KP), 3))), dtype=float)
    boxes = np.asarray(cache.get("track_boxes", np.empty((0, 4))), dtype=float)
    candidates = np.asarray(cache.get("candidate_counts", np.empty((0,))), dtype=float)
    if len(kps) == 0:
        return {"mean_conf": 0.0, "visible_ratio": 0.0, "median_bbox_area_ratio": 0.0, "mean_candidate_count": 0.0}
    frame_conf = np.nanmean(kps[:, :, 2], axis=1)
    visible_ratio = float((frame_conf > az.cfg.conf_thr).mean())
    mean_conf = float(np.nanmean(frame_conf))
    if len(boxes):
        valid = np.isfinite(boxes).all(axis=1) & ((boxes[:, 2] - boxes[:, 0]) > 4) & ((boxes[:, 3] - boxes[:, 1]) > 4)
        if valid.any():
            area = (boxes[valid, 2] - boxes[valid, 0]) * (boxes[valid, 3] - boxes[valid, 1])
            median_area_ratio = float(np.median(area) / max(float(width * height), 1.0))
        else:
            median_area_ratio = 0.0
    else:
        median_area_ratio = 0.0
    return {
        "mean_conf": mean_conf,
        "visible_ratio": visible_ratio,
        "median_bbox_area_ratio": median_area_ratio,
        "mean_candidate_count": float(np.nanmean(candidates)) if len(candidates) else 0.0,
    }


def _v42_should_use_auto_roi(diag: dict, roi_info: dict) -> tuple:
    if roi_info.get("zoom_factor", 1.0) < V42_AUTO_ROI_MIN_ZOOM_FACTOR:
        return False, "skip: roi would barely zoom"
    if diag["visible_ratio"] < V42_AUTO_ROI_VISIBLE_RATIO_THR or diag["mean_conf"] < V42_AUTO_ROI_MEAN_CONF_THR:
        return True, "weak_full_frame_tracking"
    if diag["median_bbox_area_ratio"] < V42_AUTO_ROI_SMALL_BBOX_THR and diag["mean_conf"] < V42_AUTO_ROI_SMALL_BBOX_CONF_THR:
        return True, "small_bbox_with_not_high_conf"
    return False, "skip: full-frame tracking looks stable"


def _v42_render_roi_debug_video(src: pathlib.Path, dst: pathlib.Path, roi_px: tuple):
    cap = cv2.VideoCapture(str(src))
    if not cap.isOpened():
        raise RuntimeError(f"Не могу открыть видео: {src}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*"mp4v"), float(fps), (width, height))
    x1, y1, x2, y2 = [int(v) for v in roi_px]
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 3)
        writer.write(frame)
    cap.release()
    writer.release()


def _v42_restore_kps_from_zoom_to_original(kps_zoom: np.ndarray, roi_px: tuple, frame_width: int, frame_height: int) -> np.ndarray:
    x1, y1, x2, y2 = [float(v) for v in roi_px]
    crop_w = max(1.0, x2 - x1)
    crop_h = max(1.0, y2 - y1)
    restored = np.asarray(kps_zoom, dtype=np.float32).copy()
    restored[:, 0] = x1 + (restored[:, 0] / max(float(frame_width), 1.0)) * crop_w
    restored[:, 1] = y1 + (restored[:, 1] / max(float(frame_height), 1.0)) * crop_h
    restored[:, 0] = np.clip(restored[:, 0], 0.0, float(frame_width - 1))
    restored[:, 1] = np.clip(restored[:, 1], 0.0, float(frame_height - 1))
    return restored


def _v42_run_inference_auto_roi_restored(name: str, video_path: pathlib.Path, roi_px: tuple, force: bool = False) -> dict:
    video_path = pathlib.Path(video_path)
    cache_path = OUTPUT_DIR / f"{name}_pose.npz"
    if cache_path.exists() and not force:
        cached = az._load_cached(name, cache_path)
        if cached is not None and "roi_restored_to_original_coordinates" in cached:
            az._pose_cache[name] = cached
            print(f"  {name}: {cached['kps'].shape[0]} frames (cached restored ROI pose)")
            return cached

    if az.model is None:
        az.load_model()
    infer_device = az._resolve_infer_device() if hasattr(az, "_resolve_infer_device") else getattr(az.cfg, "infer_device", "0")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Не могу открыть видео: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    x1, y1, x2, y2 = [int(v) for v in roi_px]
    x1, y1 = max(0, min(width - 2, x1)), max(0, min(height - 2, y1))
    x2, y2 = max(x1 + 2, min(width, x2)), max(y1 + 2, min(height, y2))
    roi_px = (x1, y1, x2, y2)

    kps_list = []
    track_boxes = []
    track_scores = []
    candidate_counts = []
    prev_state = None
    miss_count = 0

    print(f"  {name}: ROI-assisted inference device={infer_device}, imgsz={az.cfg.infer_imgsz}, roi={roi_px}")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        crop = frame[y1:y2, x1:x2]
        zoomed = cv2.resize(crop, (width, height), interpolation=cv2.INTER_LINEAR)
        res = az.model(zoomed, verbose=False, conf=az.cfg.conf_thr, imgsz=az.cfg.infer_imgsz, device=infer_device)[0]
        if res.keypoints is not None and len(res.keypoints.xy) > 0:
            xy_zoom_all = res.keypoints.xy.cpu().numpy()
            conf_all = res.keypoints.conf.cpu().numpy() if res.keypoints.conf is not None else np.ones((len(xy_zoom_all), len(KP)), dtype=np.float32)
            xy_all = np.stack([_v42_restore_kps_from_zoom_to_original(xy_zoom_all[i], roi_px, width, height) for i in range(len(xy_zoom_all))], axis=0)
            boxes = np.array([
                az._bbox_from_keypoints(np.concatenate([xy_all[i], conf_all[i][:, None]], axis=1))
                for i in range(len(xy_all))
            ], dtype=np.float32)
            chosen_idx, meta = az._select_primary_athlete(frame.shape, xy_all, conf_all, boxes, prev_state)
            kp = np.concatenate([xy_all[chosen_idx], conf_all[chosen_idx][:, None]], axis=1).astype(np.float32)
            box = boxes[chosen_idx].astype(np.float32)
            prev_state = {"center": meta["center"], "box": box}
            miss_count = 0
            candidate_counts.append(len(xy_all))
            track_boxes.append(box)
            track_scores.append(meta["score"])
        else:
            kp = np.zeros((len(KP), 3), dtype=np.float32)
            candidate_counts.append(0)
            track_boxes.append(np.array([np.nan, np.nan, np.nan, np.nan], dtype=np.float32))
            track_scores.append(0.0)
            miss_count += 1
            if miss_count >= int(az.cfg.tracker["max_track_misses"]):
                prev_state = None
        kps_list.append(kp)
    cap.release()

    data = {
        "kps": np.asarray(kps_list, dtype=np.float32),
        "fps": np.float32(fps),
        "track_boxes": np.asarray(track_boxes, dtype=np.float32),
        "track_scores": np.asarray(track_scores, dtype=np.float32),
        "candidate_counts": np.asarray(candidate_counts, dtype=np.int16),
        "video_path": str(video_path),
        "infer_imgsz": np.int32(az.cfg.infer_imgsz),
        "roi_px": np.asarray(roi_px, dtype=np.int32),
        "roi_restored_to_original_coordinates": np.asarray([1], dtype=np.int8),
    }
    np.savez(cache_path, **data)
    az._pose_cache[name] = data
    return data


def _v42_prepare_auto_roi_specs(run_name: str, source_dir: pathlib.Path, run_config: list, group_name: str, common_sync_sec: float | None = None, force_rebuild_pose: bool = False):
    prep_dir = OSMISLYNIE_ROOT / f"{run_name}_v42_roi_restored_prepared_inputs"
    if prep_dir.exists():
        shutil.rmtree(prep_dir)
    raw_cut_dir = prep_dir / "raw_input_cuts"
    debug_dir = prep_dir / "roi_debug_videos"
    for out_dir in [raw_cut_dir, debug_dir]:
        out_dir.mkdir(parents=True, exist_ok=True)

    cams = []
    print("V4.2 step 1: raw cuts + first-pass ROI decision")
    for cfg in run_config:
        src = _find_video(cfg["file"], source_dir)
        raw_cut = raw_cut_dir / f"{_safe_name(cfg['label'])}_raw_cut_{cfg['start_sec']:g}_{cfg['end_sec']:g}s.mp4"
        meta = _cut_interval(src, raw_cut, cfg["start_sec"], cfg["end_sec"])
        raw_name = f"{group_name}_v42_raw_{cfg['name']}"
        az.run_inference(raw_name, raw_cut, force=force_rebuild_pose)
        roi_info = _v42_compute_auto_roi_from_cache(raw_name, raw_cut)
        diag = _v42_first_pass_diagnostics(raw_name, raw_cut)
        use_auto_roi, decision_reason = _v42_should_use_auto_roi(diag, roi_info)
        if not V42_USE_CONDITIONAL_ROI:
            use_auto_roi = True
            decision_reason = "forced_auto_roi"
        if not use_auto_roi:
            width, height, _, _ = _v42_video_hw(raw_cut)
            roi_info = {
                **roi_info,
                "roi_px": (0, 0, width, height),
                "roi_norm": (0.0, 0.0, 1.0, 1.0),
                "zoom_factor": 1.0,
                "reason": "full_frame_kept",
            }
        debug_video = debug_dir / f"{_safe_name(cfg['label'])}_v42_roi_decision_on_original.mp4"
        _v42_render_roi_debug_video(raw_cut, debug_video, roi_info["roi_px"])
        cam = {
            **cfg,
            **meta,
            "raw_cut": raw_cut,
            "debug_video": debug_video,
            "video_name": f"{group_name}_v42_{cfg['name']}",
            "roi_px": roi_info["roi_px"],
            "roi_norm": roi_info["roi_norm"],
            "zoom_factor": roi_info["zoom_factor"],
            "valid_box_ratio": roi_info["valid_box_ratio"],
            "roi_reason": roi_info["reason"],
            "use_auto_roi": bool(use_auto_roi),
            "decision": decision_reason,
            "mean_conf": diag["mean_conf"],
            "visible_ratio": diag["visible_ratio"],
            "bbox_area_ratio": diag["median_bbox_area_ratio"],
            "mean_candidate_count": diag["mean_candidate_count"],
        }
        cams.append(cam)
        print(f"  {cfg['label']}: use_auto_roi={use_auto_roi} decision={decision_reason} roi={tuple(round(v,3) for v in roi_info['roi_norm'])} zoom={roi_info['zoom_factor']:.2f}")

    if common_sync_sec is None:
        common_sync_sec = min(cam["duration"] for cam in cams)
    common_sync_sec = min(float(common_sync_sec), min(cam["duration"] for cam in cams))
    common_sync_sec = float(np.floor(common_sync_sec * 10) / 10)
    return prep_dir, cams, common_sync_sec


def build_multicam_v42_roi_restored_observability_threshold_feature(group_name: str, group_spec: dict, conf_threshold: float = V41_CONF_THRESHOLD, force: bool = False) -> tuple:
    camera_features = {}
    camera_meta = []
    for spec in group_spec["cameras"]:
        cam_name = f"{group_name}_{spec['name']}"
        _v42_run_inference_auto_roi_restored(cam_name, pathlib.Path(spec["path"]), tuple(spec["roi_px"]), force=force)
        feat = az.extract(cam_name)
        camera_features[cam_name] = feat
        all_conf = np.asarray(az._pose_cache[cam_name]["kps"][:, :, 2], dtype=float)
        camera_meta.append({
            "name": cam_name,
            "spec": spec,
            "frames": int(feat["n_frames"]),
            "events": int(len(feat.get("events", []))),
            "track_score": float(feat["tracking"]["mean_track_score"]),
            "keypoint_conf_mean": float(np.nanmean(all_conf)),
            "keypoint_conf_p25": float(np.nanpercentile(all_conf, 25)),
        })

    sync_start = float(group_spec.get("sync_start_sec", 0.0))
    sync_end = float(group_spec.get("sync_end_sec"))
    target_n = int(round(np.median([(sync_end - sync_start) * float(camera_features[m["name"]]["fps"]) for m in camera_meta])))
    target_n = max(8, target_n)

    phase = {}
    phase_std = {}
    phase_effective_weight_mean = {}
    phase_valid_ratio = {}
    for ch in az.PHASE_CHANNELS:
        curves = []
        raw_weights_list = []
        confs_list = []
        for meta in camera_meta:
            feat = camera_features[meta["name"]]
            spec = meta["spec"]
            curves.append(_resample_to_n(np.asarray(feat["phase"][ch])[_sync_slice(feat, spec, sync_start, sync_end)], target_n))
            conf, _, raw_weight = _v4_weight_curve(meta["name"], feat, spec, ch, sync_start, sync_end, target_n, is_quality=False)
            confs_list.append(conf)
            raw_weights_list.append(raw_weight)
        stack = np.vstack(curves)
        confs = np.vstack(confs_list)
        raw_weights = np.vstack(raw_weights_list)
        weights = _v41_threshold_weights_with_fallback(raw_weights, confs, conf_threshold)
        if float(weights.sum()) <= 1e-8:
            weights = np.ones_like(stack)
        phase[ch] = _weighted_mean_stack(stack, weights)
        phase_std[ch] = _weighted_std_stack(stack, phase[ch], weights)
        phase_effective_weight_mean[ch] = dict(zip([m["name"] for m in camera_meta], np.mean(weights, axis=1).tolist()))
        phase_valid_ratio[ch] = dict(zip([m["name"] for m in camera_meta], np.mean(confs >= conf_threshold, axis=1).tolist()))

    fallback_quality = {}
    quality_std = {}
    quality_effective_weight = {}
    for ch in az.QUALITY_CHANNELS:
        vals = np.asarray([float(camera_features[m["name"]]["quality"][ch]) for m in camera_meta], dtype=float)
        scalar_weights = []
        for meta in camera_meta:
            conf, _, raw_weight = _v4_weight_curve(
                meta["name"], camera_features[meta["name"]], meta["spec"], ch, sync_start, sync_end, target_n, is_quality=True
            )
            weight_curve = np.where(conf >= conf_threshold, raw_weight, 0.0)
            if float(weight_curve.sum()) <= 1e-8:
                weight_curve = raw_weight
            scalar_weights.append(float(np.percentile(weight_curve, 25)))
        weights = np.asarray(scalar_weights, dtype=float)
        if float(weights.sum()) <= 1e-8:
            weights = np.ones_like(vals)
        fallback_quality[ch] = float(np.average(vals, weights=weights))
        quality_std[ch] = float(np.sqrt(np.average((vals - fallback_quality[ch]) ** 2, weights=weights)))
        quality_effective_weight[ch] = dict(zip([m["name"] for m in camera_meta], weights.tolist()))

    quality, events, peaks, troughs = _quality_from_fused_phase(
        phase=phase,
        fps=float(np.median([float(camera_features[m["name"]]["fps"]) for m in camera_meta])),
        fallback_quality=fallback_quality,
    )
    first = next(iter(camera_features.values()))
    fused_feat = {
        **{k: v for k, v in first.items() if k not in {"phase", "quality", "events", "events_peaks", "events_troughs"}},
        "n_frames": target_n,
        "fps": float(np.median([float(camera_features[m["name"]]["fps"]) for m in camera_meta])),
        "phase": phase,
        "quality": quality,
        "events": events,
        "events_peaks": peaks,
        "events_troughs": troughs,
        "tracking": {
            "mean_track_score": float(np.mean([m["track_score"] for m in camera_meta])),
            "mean_candidate_count": float(np.mean([camera_features[m["name"]]["tracking"]["mean_candidate_count"] for m in camera_meta])),
            "camera_count": len(camera_meta),
        },
        "multicam_v42": {
            "camera_meta": camera_meta,
            "fusion_weighting": "ROI-assisted pose restored to original coordinates + V4.1 weights",
            "manual_view_labels_used": False,
            "conf_threshold": float(conf_threshold),
            "sync_start_sec": sync_start,
            "sync_end_sec": sync_end,
            "phase_std": phase_std,
            "quality_std": quality_std,
            "phase_effective_weight_mean": phase_effective_weight_mean,
            "phase_valid_ratio": phase_valid_ratio,
            "quality_effective_weight": quality_effective_weight,
        },
    }
    return fused_feat, camera_features


In [ ]:
V43_MAX_ASSIGN_DIST_RATIO = 0.16


V43_MAX_TRACK_GAP = 8


V43_MIN_TRACK_FRAMES_RATIO = 0.18


V43_MULTIPLE_CANDIDATES_THR = 1.15


def _v43_torso_center_from_xy_conf(xy: np.ndarray, conf: np.ndarray) -> np.ndarray:
    ids = np.asarray(TORSO_IDS, dtype=int)
    w = np.clip(np.asarray(conf, dtype=float)[ids], 0.0, 1.0)
    pts = np.asarray(xy, dtype=float)[ids]
    if float(w.sum()) <= 1e-6:
        return np.nanmean(pts, axis=0)
    return (pts * w[:, None]).sum(axis=0) / (w.sum() + 1e-8)


def _v43_detect_candidates_by_frame(video_path: pathlib.Path):
    if az.model is None:
        az.load_model()
    infer_device = az._resolve_infer_device() if hasattr(az, "_resolve_infer_device") else getattr(az.cfg, "infer_device", "0")
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Не могу открыть видео: {video_path}")
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 1)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    diag = float(np.hypot(width, height)) + 1e-8
    frames = []
    frame_idx = 0
    print(f"  active-track first pass: device={infer_device}, imgsz={az.cfg.infer_imgsz}")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        res = az.model(frame, verbose=False, conf=az.cfg.conf_thr, imgsz=az.cfg.infer_imgsz, device=infer_device)[0]
        candidates = []
        if res.keypoints is not None and len(res.keypoints.xy) > 0:
            xy_all = res.keypoints.xy.cpu().numpy()
            conf_all = res.keypoints.conf.cpu().numpy() if res.keypoints.conf is not None else np.ones((len(xy_all), len(KP)), dtype=np.float32)
            if getattr(res, "boxes", None) is not None and len(res.boxes) == len(xy_all):
                boxes = res.boxes.xyxy.cpu().numpy().astype(np.float32)
            else:
                boxes = np.array([
                    az._bbox_from_keypoints(np.concatenate([xy_all[i], conf_all[i][:, None]], axis=1))
                    for i in range(len(xy_all))
                ], dtype=np.float32)
            for i in range(len(xy_all)):
                xy = xy_all[i].astype(np.float32)
                conf = conf_all[i].astype(np.float32)
                box = boxes[i].astype(np.float32)
                center = _v43_torso_center_from_xy_conf(xy, conf).astype(np.float32)
                candidates.append({
                    "frame": frame_idx,
                    "xy": xy,
                    "conf": conf,
                    "box": box,
                    "center": center,
                    "pose_conf": float(np.nanmean(conf)),
                })
        frames.append(candidates)
        frame_idx += 1
    cap.release()
    return frames, {"width": width, "height": height, "fps": float(fps), "diag": diag, "frames": frame_idx}


def _v43_new_track(track_id: int, cand: dict) -> dict:
    return {
        "id": track_id,
        "frames": [int(cand["frame"])],
        "centers": [np.asarray(cand["center"], dtype=np.float32)],
        "boxes": [np.asarray(cand["box"], dtype=np.float32)],
        "xys": [np.asarray(cand["xy"], dtype=np.float32)],
        "confs": [np.asarray(cand["conf"], dtype=np.float32)],
        "pose_confs": [float(cand["pose_conf"])],
        "last_frame": int(cand["frame"]),
    }


def _v43_append_track(track: dict, cand: dict):
    track["frames"].append(int(cand["frame"]))
    track["centers"].append(np.asarray(cand["center"], dtype=np.float32))
    track["boxes"].append(np.asarray(cand["box"], dtype=np.float32))
    track["xys"].append(np.asarray(cand["xy"], dtype=np.float32))
    track["confs"].append(np.asarray(cand["conf"], dtype=np.float32))
    track["pose_confs"].append(float(cand["pose_conf"]))
    track["last_frame"] = int(cand["frame"])


def _v43_build_tracklets(candidates_by_frame: list, frame_diag: float):
    tracks = []
    active = []
    next_id = 0
    max_dist = frame_diag * V43_MAX_ASSIGN_DIST_RATIO

    for frame_idx, candidates in enumerate(candidates_by_frame):
        still_active = []
        for track in active:
            if frame_idx - int(track["last_frame"]) <= V43_MAX_TRACK_GAP:
                still_active.append(track)
        active = still_active

        assigned_candidates = set()
        # Greedy matching from older active tracks to nearest current candidate by torso center.
        for track in list(active):
            if not candidates:
                continue
            last_center = np.asarray(track["centers"][-1], dtype=float)
            best_j = None
            best_dist = float("inf")
            for j, cand in enumerate(candidates):
                if j in assigned_candidates:
                    continue
                center = np.asarray(cand["center"], dtype=float)
                if not np.all(np.isfinite(center)) or not np.all(np.isfinite(last_center)):
                    continue
                dist = float(np.linalg.norm(center - last_center))
                if dist < best_dist:
                    best_dist = dist
                    best_j = j
            if best_j is not None and best_dist <= max_dist:
                _v43_append_track(track, candidates[best_j])
                assigned_candidates.add(best_j)

        for j, cand in enumerate(candidates):
            if j in assigned_candidates:
                continue
            track = _v43_new_track(next_id, cand)
            next_id += 1
            tracks.append(track)
            active.append(track)
    return tracks


def _v43_track_score(track: dict, video_meta: dict) -> dict:
    frames = np.asarray(track["frames"], dtype=int)
    centers = np.asarray(track["centers"], dtype=float)
    boxes = np.asarray(track["boxes"], dtype=float)
    xys = np.asarray(track["xys"], dtype=float)
    confs = np.asarray(track["confs"], dtype=float)
    n_total = max(int(video_meta.get("frames", 1)), 1)
    diag = float(video_meta.get("diag", 1.0)) + 1e-8

    duration_ratio = float(len(frames) / n_total)
    pose_conf = float(np.nanmean(track["pose_confs"])) if len(track["pose_confs"]) else 0.0
    valid_boxes = np.isfinite(boxes).all(axis=1) & ((boxes[:, 2] - boxes[:, 0]) > 4) & ((boxes[:, 3] - boxes[:, 1]) > 4)
    if valid_boxes.any():
        bbox_area = (boxes[valid_boxes, 2] - boxes[valid_boxes, 0]) * (boxes[valid_boxes, 3] - boxes[valid_boxes, 1])
        bbox_area_ratio = float(np.nanmedian(bbox_area) / max(float(video_meta["width"] * video_meta["height"]), 1.0))
    else:
        bbox_area_ratio = 0.0

    if len(centers) > 1:
        center_speed = np.linalg.norm(np.diff(centers, axis=0), axis=1) / diag
        center_motion_score = float(np.nanmedian(center_speed))
    else:
        center_motion_score = 0.0

    if len(xys) > 2:
        torso = 0.5 * (
            0.5 * (xys[:, KP["l_shoulder"], :2] + xys[:, KP["r_shoulder"], :2]) +
            0.5 * (xys[:, KP["l_hip"], :2] + xys[:, KP["r_hip"], :2])
        )
        scale = _v4_body_scale(np.concatenate([xys[:, :, :2], confs[:, :, None]], axis=2))
        lower_ids = [KP["l_hip"], KP["r_hip"], KP["l_knee"], KP["r_knee"], KP["l_ankle"], KP["r_ankle"]]
        rel_lower = xys[:, lower_ids, :2] - torso[:, None, :]
        rel_speed = np.linalg.norm(np.diff(rel_lower, axis=0), axis=2) / (scale[1:, None] + 1e-8)
        internal_motion = float(np.nanmedian(rel_speed)) if rel_speed.size else 0.0
        ankle_dist = np.linalg.norm(xys[:, KP["l_ankle"], :2] - xys[:, KP["r_ankle"], :2], axis=1) / (scale + 1e-8)
        ankle_range = float(np.nanpercentile(ankle_dist, 90) - np.nanpercentile(ankle_dist, 10)) if len(ankle_dist) else 0.0
    else:
        internal_motion = 0.0
        ankle_range = 0.0

    score = (
        0.28 * float(_v4_ramp(duration_ratio, V43_MIN_TRACK_FRAMES_RATIO, 0.75)) +
        0.22 * float(_v4_ramp(pose_conf, 0.35, 0.80)) +
        0.22 * float(_v4_ramp(internal_motion, 0.002, 0.040)) +
        0.18 * float(_v4_ramp(ankle_range, 0.015, 0.300)) +
        0.10 * float(_v4_ramp(center_motion_score, 0.001, 0.030))
    )
    return {
        "track_id": int(track["id"]),
        "score": float(score),
        "frames": int(len(frames)),
        "duration_ratio": duration_ratio,
        "pose_conf": pose_conf,
        "bbox_area_ratio": bbox_area_ratio,
        "internal_motion": internal_motion,
        "ankle_range": ankle_range,
        "center_motion": center_motion_score,
    }


def _v43_select_active_tracklet(candidates_by_frame: list, video_meta: dict) -> tuple:
    tracks = _v43_build_tracklets(candidates_by_frame, video_meta["diag"])
    if not tracks:
        return None, pd.DataFrame()
    rows = [_v43_track_score(track, video_meta) for track in tracks]
    df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    best_id = int(df.iloc[0]["track_id"])
    best_track = next(track for track in tracks if int(track["id"]) == best_id)
    return best_track, df


def _v43_roi_from_active_track(track: dict, video_meta: dict) -> dict:
    width = int(video_meta["width"])
    height = int(video_meta["height"])
    if track is None:
        return {
            "roi_px": (0, 0, width, height),
            "roi_norm": (0.0, 0.0, 1.0, 1.0),
            "valid_box_ratio": 0.0,
            "zoom_factor": 1.0,
            "reason": "no_active_track",
        }
    boxes = np.asarray(track["boxes"], dtype=float)
    valid = np.isfinite(boxes).all(axis=1) & ((boxes[:, 2] - boxes[:, 0]) > 4) & ((boxes[:, 3] - boxes[:, 1]) > 4)
    valid_ratio = float(valid.mean()) if len(valid) else 0.0
    if valid_ratio < V42_AUTO_ROI_MIN_VALID_BOX_RATIO:
        return {
            "roi_px": (0, 0, width, height),
            "roi_norm": (0.0, 0.0, 1.0, 1.0),
            "valid_box_ratio": valid_ratio,
            "zoom_factor": 1.0,
            "reason": "active_track_too_few_valid_boxes",
        }
    vb = boxes[valid]
    x1 = float(np.percentile(vb[:, 0], 5))
    y1 = float(np.percentile(vb[:, 1], 5))
    x2 = float(np.percentile(vb[:, 2], 95))
    y2 = float(np.percentile(vb[:, 3], 95))
    x1, y1, x2, y2 = _v42_expand_roi_keep_aspect(x1, y1, x2, y2, width, height, margin=V42_AUTO_ROI_MARGIN)
    roi_px = (int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2)))
    crop_w = max(1, roi_px[2] - roi_px[0])
    crop_h = max(1, roi_px[3] - roi_px[1])
    zoom_factor = float(min(width / crop_w, height / crop_h))
    return {
        "roi_px": roi_px,
        "roi_norm": (roi_px[0] / width, roi_px[1] / height, roi_px[2] / width, roi_px[3] / height),
        "valid_box_ratio": valid_ratio,
        "zoom_factor": zoom_factor,
        "reason": "active_tracklet_roi",
    }


def _v43_should_use_active_roi(diag: dict, roi_info: dict, active_meta: dict) -> tuple:
    if roi_info.get("zoom_factor", 1.0) < V42_AUTO_ROI_MIN_ZOOM_FACTOR:
        return False, "skip: active ROI would barely zoom"
    if diag.get("mean_candidate_count", 0.0) >= V43_MULTIPLE_CANDIDATES_THR:
        return True, "multiple_candidates_use_active_tracklet_roi"
    if active_meta and active_meta.get("duration_ratio", 0.0) >= V43_MIN_TRACK_FRAMES_RATIO:
        use_roi, reason = _v42_should_use_auto_roi(diag, roi_info)
        if use_roi:
            return True, "active_tracklet_" + reason
    return _v42_should_use_auto_roi(diag, roi_info)


def _v43_prepare_active_roi_specs(run_name: str, source_dir: pathlib.Path, run_config: list, group_name: str, common_sync_sec: float | None = None, force_rebuild_pose: bool = False):
    prep_dir = OSMISLYNIE_ROOT / f"{run_name}_v43_active_roi_prepared_inputs"
    if prep_dir.exists():
        shutil.rmtree(prep_dir)
    raw_cut_dir = prep_dir / "raw_input_cuts"
    debug_dir = prep_dir / "roi_debug_videos"
    tracklet_dir = prep_dir / "active_tracklet_tables"
    for out_dir in [raw_cut_dir, debug_dir, tracklet_dir]:
        out_dir.mkdir(parents=True, exist_ok=True)

    cams = []
    print("V4.3 step 1: raw cuts + active-athlete ROI decision")
    for cfg in run_config:
        src = _find_video(cfg["file"], source_dir)
        raw_cut = raw_cut_dir / f"{_safe_name(cfg['label'])}_raw_cut_{cfg['start_sec']:g}_{cfg['end_sec']:g}s.mp4"
        meta = _cut_interval(src, raw_cut, cfg["start_sec"], cfg["end_sec"])
        raw_name = f"{group_name}_v43_raw_{cfg['name']}"
        az.run_inference(raw_name, raw_cut, force=force_rebuild_pose)
        diag = _v42_first_pass_diagnostics(raw_name, raw_cut)
        candidates_by_frame, video_meta = _v43_detect_candidates_by_frame(raw_cut)
        active_track, tracklet_df = _v43_select_active_tracklet(candidates_by_frame, video_meta)
        if len(tracklet_df):
            tracklet_df.to_csv(tracklet_dir / f"{_safe_name(cfg['label'])}_active_tracklets.csv", index=False)
            active_meta = tracklet_df.iloc[0].to_dict()
        else:
            active_meta = {}
        roi_info = _v43_roi_from_active_track(active_track, video_meta)
        if roi_info["reason"] == "no_active_track":
            roi_info = _v42_compute_auto_roi_from_cache(raw_name, raw_cut)
            roi_info = {**roi_info, "reason": "fallback_default_track_roi"}
        use_auto_roi, decision_reason = _v43_should_use_active_roi(diag, roi_info, active_meta)
        if not V42_USE_CONDITIONAL_ROI:
            use_auto_roi = True
            decision_reason = "forced_active_roi"
        if not use_auto_roi:
            width, height, _, _ = _v42_video_hw(raw_cut)
            roi_info = {
                **roi_info,
                "roi_px": (0, 0, width, height),
                "roi_norm": (0.0, 0.0, 1.0, 1.0),
                "zoom_factor": 1.0,
                "reason": "full_frame_kept",
            }
        debug_video = debug_dir / f"{_safe_name(cfg['label'])}_v43_active_roi_decision_on_original.mp4"
        _v42_render_roi_debug_video(raw_cut, debug_video, roi_info["roi_px"])
        cam = {
            **cfg,
            **meta,
            "raw_cut": raw_cut,
            "debug_video": debug_video,
            "video_name": f"{group_name}_v43_{cfg['name']}",
            "roi_px": roi_info["roi_px"],
            "roi_norm": roi_info["roi_norm"],
            "zoom_factor": roi_info["zoom_factor"],
            "valid_box_ratio": roi_info["valid_box_ratio"],
            "roi_reason": roi_info["reason"],
            "use_auto_roi": bool(use_auto_roi),
            "decision": decision_reason,
            "mean_conf": diag["mean_conf"],
            "visible_ratio": diag["visible_ratio"],
            "bbox_area_ratio": diag["median_bbox_area_ratio"],
            "mean_candidate_count": diag["mean_candidate_count"],
            "active_track_id": int(active_meta.get("track_id", -1)) if active_meta else -1,
            "active_track_score": float(active_meta.get("score", 0.0)) if active_meta else 0.0,
            "active_track_frames": int(active_meta.get("frames", 0)) if active_meta else 0,
            "active_track_duration_ratio": float(active_meta.get("duration_ratio", 0.0)) if active_meta else 0.0,
        }
        cams.append(cam)
        print(
            f"  {cfg['label']}: use_auto_roi={use_auto_roi} decision={decision_reason} "
            f"active_score={cam['active_track_score']:.3f} roi={tuple(round(v,3) for v in roi_info['roi_norm'])} zoom={roi_info['zoom_factor']:.2f}"
        )

    if common_sync_sec is None:
        common_sync_sec = min(cam["duration"] for cam in cams)
    common_sync_sec = min(float(common_sync_sec), min(cam["duration"] for cam in cams))
    common_sync_sec = float(np.floor(common_sync_sec * 10) / 10)
    return prep_dir, cams, common_sync_sec


def run_multicam_v43_active_roi_observability_experiment(run_name: str, source_dir: pathlib.Path, run_config: list, group_name: str, common_sync_sec: float | None = None, force_rebuild_pose: bool = False, conf_threshold: float = V41_CONF_THRESHOLD):
    prep_dir, cams, common_sync_sec = _v43_prepare_active_roi_specs(
        run_name, source_dir, run_config, group_name, common_sync_sec=common_sync_sec, force_rebuild_pose=force_rebuild_pose
    )
    run_dir = OSMISLYNIE_ROOT / f"{run_name}_v43_active_roi_auto_observability_threshold"
    if run_dir.exists():
        shutil.rmtree(run_dir)
    overlay_dir = run_dir / "overlays"
    graph_dir = run_dir / "graphs"
    for out_dir in [overlay_dir, graph_dir]:
        out_dir.mkdir(parents=True, exist_ok=True)

    group_spec = {
        "sync_start_sec": 0.0,
        "sync_end_sec": common_sync_sec,
        "cameras": [
            {"name": cam["name"], "path": cam["raw_cut"], "offset_sec": 0.0, "roi_px": cam["roi_px"]}
            for cam in cams
        ],
    }

    print("\nV4.3 step 2: active-athlete ROI pose restored to original coordinates + V4.1 fusion")
    fused_feat, single_feats = build_multicam_v42_roi_restored_observability_threshold_feature(
        f"{group_name}_v43", group_spec, conf_threshold=conf_threshold, force=force_rebuild_pose
    )

    merged = {**features, **single_feats, f"{group_name}_v43": fused_feat}
    report = az.reference_quality_report(merged, reference_scores=az.cfg.reference_scores)

    rows = []
    for cam in cams:
        video_name = f"{group_name}_v43_{cam['name']}"
        row = report[report["video"] == video_name].reset_index(drop=True).iloc[0]
        feat = single_feats[video_name]
        conf = np.asarray(az._pose_cache[video_name]["kps"][:, :, 2], dtype=float)
        rows.append({
            "label": cam["label"],
            "kind": "single_camera_active_roi_restored",
            "score": float(row["quality_score_100"]),
            "closest_reference": row["closest_reference"],
            "frames": int(feat["n_frames"]),
            "events": int(len(feat.get("events", []))),
            "track": float(feat["tracking"]["mean_track_score"]),
            "mean_conf": float(np.nanmean(conf)),
            "visible_ratio": float((np.nanmean(conf, axis=1) > az.cfg.conf_thr).mean()),
            "use_auto_roi": cam["use_auto_roi"],
            "zoom_factor": float(cam["zoom_factor"]),
            "active_track_score": float(cam["active_track_score"]),
        })
    fused_row = report[report["video"] == f"{group_name}_v43"].reset_index(drop=True).iloc[0]
    rows.append({
        "label": "3cam fused V4.3",
        "kind": "multicam_fused_v43_active_roi_auto_observability_threshold",
        "score": float(fused_row["quality_score_100"]),
        "closest_reference": fused_row["closest_reference"],
        "frames": int(fused_feat["n_frames"]),
        "events": int(len(fused_feat.get("events", []))),
        "track": float(fused_feat["tracking"]["mean_track_score"]),
        "mean_conf": np.nan,
        "visible_ratio": np.nan,
        "use_auto_roi": np.nan,
        "zoom_factor": np.nan,
        "active_track_score": np.nan,
    })
    score_df = pd.DataFrame(rows)
    roi_df = pd.DataFrame([{k: cam[k] for k in [
        "label", "use_auto_roi", "decision", "mean_conf", "visible_ratio", "bbox_area_ratio",
        "mean_candidate_count", "active_track_id", "active_track_score", "active_track_frames",
        "active_track_duration_ratio", "roi_norm", "zoom_factor", "roi_reason"
    ]} for cam in cams])

    print("\nV4.3 active ROI decision summary:")
    print(roi_df.round(3).to_string(index=False))
    print("\nSingle-camera vs 3cam fused V4.3:")
    print(score_df[["label", "score", "closest_reference", "frames", "events", "track", "mean_conf", "visible_ratio", "use_auto_roi", "zoom_factor", "active_track_score"]].round(3).to_string(index=False))

    overlay_paths = []
    for cam in cams:
        video_name = f"{group_name}_v43_{cam['name']}"
        overlay_path = overlay_dir / f"{cam['label']}_v43_active_roi_restored_pose_overlay.mp4"
        _render_pose_overlay_video(video_name, cam["raw_cut"], overlay_path)
        overlay_paths.append(overlay_path)

    _plot_scores(score_df, graph_dir)
    # Tracking diagnostics are already present in score_df/summary.
    # Do not render a separate plot here: in some notebook runs it is easy to keep a stale empty figure output.
    _plot_feature_fusion_v41(cams, single_feats, fused_feat, common_sync_sec, graph_dir, conf_threshold=conf_threshold)

    summary_lines = [
        "V4.3 = active-athlete auto-ROI + restored original coordinates + V4.1 fusion",
        "manual_view_labels_used: False",
        f"conf_threshold: {conf_threshold:.2f}",
        "",
        "roi_decisions:",
        roi_df.round(3).to_string(index=False),
        "",
        "score_df:",
        score_df.round(3).to_string(index=False),
        "",
        f"prepared_input_dir: {prep_dir}",
        f"graphs_dir: {graph_dir}",
        "overlay_videos:",
        *[str(path) for path in overlay_paths],
    ]
    (run_dir / "RUN_SUMMARY_V43_ACTIVE_ROI_AUTO_OBSERVABILITY_THRESHOLD.txt").write_text("\n".join(summary_lines), encoding="utf-8")
    print("\nSaved V4.3 to:", run_dir)
    return {"run_dir": run_dir, "prepared_input_dir": prep_dir, "score_df": score_df, "roi_df": roi_df, "cams": cams, "fused_feat": fused_feat, "single_feats": single_feats}


## Coverage-aware final score

`raw_similarity_score` считает похожесть на reference-прокаты по всем pose-фичам.

Дополнительно считаем `coverage_factor`: насколько эти фичи реально наблюдаемы по confidence нужных keypoints. Если фича плохо видна на одной камере, камера почти не влияет на fusion. Если фича плохо видна на всех камерах, итоговый `score` дополнительно снижается.

```text
score = raw_similarity_score * coverage_factor
```


In [ ]:
# Coverage-aware score layer.
# Это не меняет low-level fusion: камеры уже объединены по confidence/observability.
# Здесь добавляется честный штраф за то, что часть pose-фич была плохо видна.


def _weighted_channel_coverage_mean(coverage: dict, channels: list, weights) -> float:
    if not channels:
        return 1.0
    vals = np.asarray([float(coverage.get(ch, coverage.get(_base_channel(ch), 1.0))) for ch in channels], dtype=float)
    if weights is None or len(weights) != len(channels):
        w = np.ones(len(channels), dtype=float) / max(len(channels), 1)
    else:
        w = np.asarray(weights, dtype=float)
        w = w / (w.sum() + 1e-8)
    return float(np.clip(np.sum(vals * w), 0.0, 1.0))


def _pose_coverage_factor(phase_coverage: dict, quality_coverage: dict) -> float:
    phase_factor = _weighted_channel_coverage_mean(
        phase_coverage,
        list(az.PHASE_CHANNELS),
        getattr(az, "_phase_w", None),
    )
    quality_factor = _weighted_channel_coverage_mean(
        quality_coverage,
        list(az.QUALITY_CHANNELS),
        getattr(az, "_quality_w", None),
    )
    alpha = float(getattr(az.cfg, "alpha", 0.5))
    return float(np.clip(alpha * phase_factor + (1.0 - alpha) * quality_factor, 0.0, 1.0))


def _pose_coverage_for_single_video(video_name: str, feat: dict, duration_sec: float, target_n: int, conf_threshold: float):
    spec = {"offset_sec": 0.0}
    phase_coverage = {}
    quality_coverage = {}
    for ch in az.PHASE_CHANNELS:
        conf = _confidence_curve(video_name, feat, spec, ch, 0.0, duration_sec, target_n, is_quality=False)
        phase_coverage[ch] = float(np.mean(conf >= conf_threshold))
    for ch in az.QUALITY_CHANNELS:
        conf = _confidence_curve(video_name, feat, spec, ch, 0.0, duration_sec, target_n, is_quality=True)
        quality_coverage[ch] = float(np.mean(conf >= conf_threshold))
    return phase_coverage, quality_coverage


def _pose_coverage_for_fused_video(cams: list, single_feats: dict, duration_sec: float, target_n: int, conf_threshold: float):
    spec = {"offset_sec": 0.0}
    phase_coverage = {}
    quality_coverage = {}
    for ch in az.PHASE_CHANNELS:
        curves = []
        for cam in cams:
            video_name = cam["video_name"]
            curves.append(_confidence_curve(video_name, single_feats[video_name], spec, ch, 0.0, duration_sec, target_n, is_quality=False))
        best_conf = np.max(np.vstack(curves), axis=0)
        phase_coverage[ch] = float(np.mean(best_conf >= conf_threshold))
    for ch in az.QUALITY_CHANNELS:
        curves = []
        for cam in cams:
            video_name = cam["video_name"]
            curves.append(_confidence_curve(video_name, single_feats[video_name], spec, ch, 0.0, duration_sec, target_n, is_quality=True))
        best_conf = np.max(np.vstack(curves), axis=0)
        quality_coverage[ch] = float(np.mean(best_conf >= conf_threshold))
    return phase_coverage, quality_coverage


def _low_coverage_features_text(phase_coverage: dict, quality_coverage: dict, limit: int = 6) -> str:
    merged = {**{f"phase:{k}": v for k, v in phase_coverage.items()}, **{f"quality:{k}": v for k, v in quality_coverage.items()}}
    low = sorted(merged.items(), key=lambda item: item[1])[:limit]
    return ", ".join(f"{name}={value:.2f}" for name, value in low)


def _apply_pose_coverage_adjustment(result: dict, conf_threshold: float = V41_CONF_THRESHOLD) -> dict:
    score_df = result["score_df"].copy()
    cams = result["cams"]
    single_feats = result["single_feats"]
    fused_feat = result["fused_feat"]
    target_n = int(fused_feat["n_frames"])
    duration_sec = float(target_n / (float(fused_feat["fps"]) + 1e-8))

    coverage_by_label = {}
    for cam in cams:
        video_name = cam["video_name"]
        phase_cov, quality_cov = _pose_coverage_for_single_video(
            video_name, single_feats[video_name], duration_sec, target_n, conf_threshold
        )
        coverage_by_label[cam["label"]] = {
            "coverage_factor": _pose_coverage_factor(phase_cov, quality_cov),
            "low_coverage_features": _low_coverage_features_text(phase_cov, quality_cov),
        }

    fused_phase_cov, fused_quality_cov = _pose_coverage_for_fused_video(
        cams, single_feats, duration_sec, target_n, conf_threshold
    )
    fused_factor = _pose_coverage_factor(fused_phase_cov, fused_quality_cov)
    fused_low = _low_coverage_features_text(fused_phase_cov, fused_quality_cov)

    raw_scores = score_df["score"].astype(float).copy()
    if "raw_similarity_score" not in score_df.columns:
        score_df.insert(score_df.columns.get_loc("score"), "raw_similarity_score", raw_scores)
    else:
        score_df["raw_similarity_score"] = raw_scores
    score_df["coverage_factor"] = 1.0
    score_df["low_coverage_features"] = ""

    for idx, row in score_df.iterrows():
        if str(row.get("kind", "")).startswith("multicam"):
            coverage_info = {"coverage_factor": fused_factor, "low_coverage_features": fused_low}
        else:
            coverage_info = coverage_by_label.get(row["label"], {"coverage_factor": 1.0, "low_coverage_features": ""})
        factor = float(coverage_info["coverage_factor"])
        score_df.loc[idx, "coverage_factor"] = factor
        score_df.loc[idx, "score"] = float(score_df.loc[idx, "raw_similarity_score"]) * factor
        score_df.loc[idx, "low_coverage_features"] = coverage_info["low_coverage_features"]
    score_df["score_delta_coverage"] = score_df["score"] - score_df["raw_similarity_score"]

    result["score_df"] = score_df
    result["coverage"] = {
        "conf_threshold": float(conf_threshold),
        "fused_phase_coverage": fused_phase_cov,
        "fused_quality_coverage": fused_quality_cov,
    }

    run_dir = pathlib.Path(result["run_dir"])
    graph_dir = run_dir / "graphs"
    graph_dir.mkdir(parents=True, exist_ok=True)
    score_df.to_csv(run_dir / "score_df_coverage_adjusted.csv", index=False)
    _plot_scores(score_df, graph_dir)

    print("\nCoverage-adjusted pose exercise score:")
    cols = [
        "label", "raw_similarity_score", "coverage_factor", "score", "closest_reference",
        "frames", "events", "track", "low_coverage_features",
    ]
    print(score_df[cols].round(3).to_string(index=False))

    summary_lines = [
        "Final pose-only exercise score = raw reference similarity * pose coverage factor",
        f"conf_threshold: {conf_threshold:.2f}",
        "",
        "score_df_coverage_adjusted:",
        score_df.round(3).to_string(index=False),
        "",
        "fused_low_coverage_features:",
        fused_low,
    ]
    (run_dir / "RUN_SUMMARY_POSE_EXERCISE_COVERAGE_ADJUSTED.txt").write_text("\n".join(summary_lines), encoding="utf-8")
    return result


_BASE_RUN_MULTICAM_V43_ACTIVE_ROI_OBSERVABILITY_EXPERIMENT = globals().get(
    "_BASE_RUN_MULTICAM_V43_ACTIVE_ROI_OBSERVABILITY_EXPERIMENT",
    run_multicam_v43_active_roi_observability_experiment,
)


def run_multicam_v43_active_roi_observability_experiment(
    run_name: str,
    source_dir: pathlib.Path,
    run_config: list,
    group_name: str,
    common_sync_sec: float | None = None,
    force_rebuild_pose: bool = False,
    conf_threshold: float = V41_CONF_THRESHOLD,
):
    result = _BASE_RUN_MULTICAM_V43_ACTIVE_ROI_OBSERVABILITY_EXPERIMENT(
        run_name=run_name,
        source_dir=source_dir,
        run_config=run_config,
        group_name=group_name,
        common_sync_sec=common_sync_sec,
        force_rebuild_pose=force_rebuild_pose,
        conf_threshold=conf_threshold,
    )
    return _apply_pose_coverage_adjustment(result, conf_threshold=conf_threshold)


In [ ]:
# Final pose-only exercise run.
# This is the version intended for GitHub: active-athlete ROI + confidence/observability fusion + coverage-aware score.

RUN_FINAL_POSE_EXERCISE_6449 = True
RUN_FINAL_POSE_EXERCISE_6448 = True

if RUN_FINAL_POSE_EXERCISE_6449:
    final_pose_exercise_6449_result = run_multicam_v43_active_roi_observability_experiment(
        **{**_with_suffix_config(MULTICAM_RUN_CONFIG, "pose_exercise_final"), "force_rebuild_pose": False, "conf_threshold": V41_CONF_THRESHOLD}
    )

if RUN_FINAL_POSE_EXERCISE_6448:
    final_pose_exercise_6448_result = run_multicam_v43_active_roi_observability_experiment(
        **{**_with_suffix_config(MULTICAM_6448_CONFIG, "pose_exercise_final"), "force_rebuild_pose": False, "conf_threshold": V41_CONF_THRESHOLD}
    )


In [ ]:
# Optional: collect final pose-overlay videos from both pose-exercise runs into one zip.
# This cell does not run inference; it only copies videos that were already generated by the final run.

WORKING_DIR = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else ROOT / "kaggle_working"
WORKING_DIR.mkdir(parents=True, exist_ok=True)

POSE_OVERLAYS_DIR = WORKING_DIR / "pose_exercise_overlays"
POSE_OVERLAYS_DIR.mkdir(parents=True, exist_ok=True)
for old_file in POSE_OVERLAYS_DIR.glob("*.mp4"):
    old_file.unlink()

RUNS_TO_COLLECT = [
    ("run6449", MULTICAM_RUN_CONFIG, "final_pose_exercise_6449_result"),
    ("run6448", MULTICAM_6448_CONFIG, "final_pose_exercise_6448_result"),
]


def _latest_existing(paths):
    paths = [pathlib.Path(p) for p in paths if pathlib.Path(p).exists()]
    if not paths:
        return None
    return max(paths, key=lambda p: p.stat().st_mtime)


def _expected_pose_exercise_run_dir(config: dict) -> pathlib.Path:
    cfg = _with_suffix_config(config, "pose_exercise_final")
    return OSMISLYNIE_ROOT / f"{cfg['run_name']}_v43_active_roi_auto_observability_threshold"


copied, missing = [], []
for run_tag, config, result_var in RUNS_TO_COLLECT:
    result = globals().get(result_var)
    run_dir = pathlib.Path(result["run_dir"]) if isinstance(result, dict) and "run_dir" in result else _expected_pose_exercise_run_dir(config)
    print(f"\n{run_tag}: searching pose overlays in {run_dir}")
    for cam in config["run_config"]:
        label = cam["label"]
        candidates = []
        candidates.append(run_dir / "overlays" / f"{label}_v43_active_roi_restored_pose_overlay.mp4")
        if run_dir.exists():
            candidates.extend(run_dir.rglob(f"*{label}*pose_overlay*.mp4"))
        src = _latest_existing(candidates)
        if src is None:
            missing.append((run_tag, label))
            print("  missing:", label)
            continue
        dst = POSE_OVERLAYS_DIR / f"{run_tag}_{label}_pose_overlay.mp4"
        shutil.copy2(src, dst)
        copied.append(dst)
        print("  copied:", dst.name)

zip_path = shutil.make_archive(str(WORKING_DIR / "pose_exercise_overlays"), "zip", root_dir=POSE_OVERLAYS_DIR)
print("\nCopied pose-overlay videos:", len(copied))
if missing:
    print("Missing overlays:", missing)
print("Saved zip:", zip_path)


## Feature glossary


In [ ]:
# Human-readable feature glossary for all pose-only exercise features.

FEATURE_GLOSSARY = [
    {"group": "phase", "feature": "ankles_distance", "meaning": "расстояние между лодыжками", "higher_means": "ноги дальше друг от друга", "lower_means": "ноги ближе друг к другу"},
    {"group": "phase", "feature": "knee_angle_mean", "meaning": "средний угол в коленях", "higher_means": "колени более разогнуты", "lower_means": "колени сильнее согнуты"},
    {"group": "phase", "feature": "knee_angle_asym", "meaning": "асимметрия углов в левом и правом колене", "higher_means": "колени работают менее симметрично", "lower_means": "колени ближе по углу, движение симметричнее"},
    {"group": "phase", "feature": "torso_lean_abs_deg", "meaning": "абсолютный наклон корпуса в градусах", "higher_means": "корпус сильнее наклонен", "lower_means": "корпус ближе к вертикали"},
    {"group": "phase", "feature": "hands_to_torso_mean", "meaning": "среднее расстояние рук от корпуса", "higher_means": "руки дальше от корпуса / шире разнесены", "lower_means": "руки компактнее около корпуса"},
    {"group": "phase", "feature": "head_pitch_rel", "meaning": "наклон головы относительно плеч", "higher_means": "голова сильнее опущена/наклонена по этому признаку", "lower_means": "голова более поднята относительно корпуса"},
    {"group": "phase", "feature": "hip_lateral_sway", "meaning": "боковое смещение таза", "higher_means": "таз сильнее гуляет вбок", "lower_means": "таз стабильнее"},
    {"group": "phase", "feature": "elbow_angle_mean", "meaning": "средний угол в локтях", "higher_means": "руки более разогнуты", "lower_means": "руки сильнее согнуты"},
    {"group": "phase", "feature": "shoulder_elevation", "meaning": "подъем/перекос плеч относительно корпуса", "higher_means": "плечи выше/перекос заметнее", "lower_means": "плечи ровнее и спокойнее"},
    {"group": "phase", "feature": "spine_angle", "meaning": "угол линии корпуса/спины", "higher_means": "спина/корпус сильнее отклонены по углу", "lower_means": "спина ближе к нейтральному положению"},
    {"group": "phase", "feature": "motion_speed_norm", "meaning": "нормированная скорость движения корпуса", "higher_means": "движение быстрее", "lower_means": "движение медленнее"},
    {"group": "phase", "feature": "motion_dir_stability_local", "meaning": "стабильность направления движения", "higher_means": "направление движения стабильнее", "lower_means": "направление дергается/меняется сильнее"},
    {"group": "phase", "feature": "body_motion_align", "meaning": "насколько корпус согласован с направлением движения", "higher_means": "корпус лучше направлен по движению", "lower_means": "корпус хуже согласован с движением"},
    {"group": "phase", "feature": "head_motion_align", "meaning": "насколько голова согласована с направлением движения", "higher_means": "голова лучше направлена по движению", "lower_means": "голова хуже согласована с движением"},
    {"group": "phase", "feature": "*_vel", "meaning": "скорость изменения соответствующей phase-фичи", "higher_means": "фича меняется быстрее/резче", "lower_means": "фича меняется плавнее/медленнее"},
    {"group": "quality", "feature": "head_up", "meaning": "агрегированная оценка положения головы", "higher_means": "лучше: голова выше/стабильнее", "lower_means": "хуже: голова сильнее опущена"},
    {"group": "quality", "feature": "torso_upright", "meaning": "агрегированная оценка вертикальности корпуса", "higher_means": "лучше: корпус ровнее", "lower_means": "хуже: корпус сильнее завален"},
    {"group": "quality", "feature": "hands_compact", "meaning": "компактность рук", "higher_means": "лучше: руки компактнее", "lower_means": "хуже: руки дальше от корпуса"},
    {"group": "quality", "feature": "tempo_stability", "meaning": "стабильность темпа циклов", "higher_means": "лучше: темп ровнее", "lower_means": "хуже: темп более рваный"},
    {"group": "quality", "feature": "leg_control", "meaning": "стабильность работы ног", "higher_means": "лучше: амплитуда ног стабильнее", "lower_means": "хуже: ноги работают более рвано"},
    {"group": "quality", "feature": "knee_symmetry", "meaning": "симметрия коленей", "higher_means": "лучше: колени симметричнее", "lower_means": "хуже: больше асимметрия"},
    {"group": "quality", "feature": "upper_body_stability", "meaning": "стабильность верхней части корпуса", "higher_means": "лучше: верх тела стабильнее", "lower_means": "хуже: верх тела больше качается"},
    {"group": "quality", "feature": "arm_discipline", "meaning": "стабильность/дисциплина рук", "higher_means": "лучше: руки двигаются спокойнее", "lower_means": "хуже: руки дергаются/разлетаются"},
    {"group": "quality", "feature": "cycle_amplitude_cv", "meaning": "разброс амплитуды циклов", "higher_means": "лучше: амплитуда циклов ровнее", "lower_means": "хуже: амплитуда скачет"},
    {"group": "quality", "feature": "peak_posture_consistency", "meaning": "похожесть позы в пиковые моменты", "higher_means": "лучше: поза в пиках стабильнее", "lower_means": "хуже: поза в пиках разная"},
    {"group": "quality", "feature": "direction_control", "meaning": "контроль направления движения", "higher_means": "лучше: направление стабильнее", "lower_means": "хуже: направление менее контролируемое"},
    {"group": "quality", "feature": "body_motion_alignment", "meaning": "согласованность корпуса с движением", "higher_means": "лучше: корпус согласован с траекторией", "lower_means": "хуже: корпус расходится с траекторией"},
    {"group": "quality", "feature": "head_motion_alignment", "meaning": "согласованность головы с движением", "higher_means": "лучше: голова согласована с траекторией", "lower_means": "хуже: голова расходится с траекторией"},
    {"group": "quality", "feature": "head_body_heading_consistency", "meaning": "согласованность направления головы и корпуса", "higher_means": "лучше: голова и корпус смотрят согласованнее", "lower_means": "хуже: голова и корпус расходятся"},
]

feature_glossary_df = pd.DataFrame(FEATURE_GLOSSARY)
print("Как читать значения:")
print("- phase: покадровые измерения; выше/ниже описывает сам сигнал, не обязательно хорошо/плохо.")
print("- quality: агрегаты для оценки; обычно выше = лучше, ниже/более отрицательно = хуже.")
print("- *_vel: скорость изменения базовой фичи; большие значения означают более резкое изменение.")
print("- финальный score не равен одной фиче: это similarity к reference-прокатам по набору фич.")

try:
    display(feature_glossary_df)
except NameError:
    print(feature_glossary_df.to_string(index=False))
